# 🎯 [K11] 군 일정 관리 AI - 일과·근무·교육 일정 충돌 점검과 개인 일정 추천 구현
### Google Colab 실습 환경 (작전기획·일정관리 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K11/K11_solution.ipynb)

---

## 📌 1. 프로젝트 개요 및 배경

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전기획·일정관리 (Level 2) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 20.0시간) |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
군 생활에서는 부대 일과, 근무, 교육·훈련, 휴가가 한 사람의 하루에 겹치고, 변경 공지가 나올 때마다 일정을 다시 맞춰야 합니다. 이 프로젝트에서는 파이썬으로 가상 한빛대대 2주간의 일정 데이터를 구조화하고, 시간 겹침과 근무 후 휴식 부족을 찾아내며, 남는 시간에 자격증 공부·체력단련 같은 개인 목표를 탐욕 방법과 최적화(scipy.optimize)로 배정하는 '군 일정 관리 도우미'를 단계별로 구현합니다. 일정 변경 공지를 반영해 다시 계산하고 주간 일정표와 알림 문구를 만들며, 과부하 상황을 시험하는 과정을 통해 AI는 제안만 하고 근무 편성과 일정 확정은 분대장·행정반이 하는 원칙을 익힙니다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 실습의 모든 데이터(공지, 규정, 일정 등)는 보안 규정을 준수하여 **코드 내 난수 및 가상 시나리오로 생성**됩니다.
> - 실제 부대 작전, 군사 기밀 및 개인 식별 정보는 일체 포함되지 않습니다.

---



In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 01. 일정 데이터 구조화하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **가상 2주 일정(일과·근무·교육·휴가)을 시작·종료 시각이 있는 표로 구조화하고 형식을 검사합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **일정을 '11/02 00:00부터의 분' 단위 표로 바꾸고 종류별 건수와 합계 시간을 확인하고 유효성을 점검하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `일정 표, 종류별 요약, 형식 검사 함수와 시험 결과` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 데이터 구조화 — 일정을 일관된 구조로 정리하고 형식 오류를 잡아내는 검사를 만들었는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(일정 표, 종류별 요약, 형식 검사 함수와 시험 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""가상 한빛대대 2주 일정(일과·근무·교육·휴가)을 시작·종료 시각이 있는 표로 구조화하고 형식 오류를 검사한다."""
import re
import datetime as dt
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 2주(11/02~11/15) 일정 자료 (실제 부대·인원·규정과 무관)
# 시각은 "일차(0=11/02 월요일) + HH:MM"으로 표기하며, 24시를 넘기면 26:00처럼 씁니다.
# ---------------------------------------------------------------------------
START = dt.date(2026, 11, 2)
DOW = "월화수목금토일"
DAY_END = "21:00"          # 개인 시간 계획은 저녁 점호(21:00) 전까지만 다룬다


def t(day, hhmm):
    # [TODO : 학생 실습 1] 일정 데이터 구조화하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def E(eid, name, kind, day, s, e, end_day=None, sub=""):
    # [TODO : 학생 실습 1] 일정 데이터 구조화하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_routine():
    ev = []
    for d in range(14):
        wd = d % 7
        if wd < 5:
            items = [("기상·점호", "fixed", "06:30", "07:00"), ("조식", "fixed", "07:00", "08:00"),
                     ("청소·정비", "fixed", "08:00", "09:00"), ("오전 일과", "work", "09:00", "12:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("오후 일과", "work", "13:00", "17:30"),
                     ("석식", "fixed", "17:30", "18:30"), ("저녁 점호", "fixed", "21:00", "21:30")]
        else:
            items = [("기상·점호", "fixed", "07:30", "08:00"), ("조식", "fixed", "08:00", "09:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("석식", "fixed", "17:30", "18:30"),
                     ("저녁 점호", "fixed", "21:00", "21:30")]
            if wd == 5:
                items.insert(2, ("주말 청소·정비", "work", "09:00", "10:30"))
        for k, (nm, sub, s, e) in enumerate(items):
            ev.append(E(f"R{d:02d}{k}", nm, "routine", d, s, e, sub=sub))
    return ev


EVENTS = make_routine() + [
    # 개인 근무표 (가상): 불침번·경계근무·당직 보조
    E("D01", "불침번", "duty", 0, "26:00", "28:00", sub="불침번"),
    E("D02", "불침번", "duty", 2, "22:00", "24:00", sub="불침번"),
    E("D03", "경계근무", "duty", 3, "16:00", "18:00", sub="경계근무"),
    E("D04", "불침번", "duty", 4, "26:00", "28:00", sub="불침번"),
    E("D05", "당직 보조", "duty", 6, "08:00", "32:00", sub="당직"),
    E("D06", "불침번", "duty", 8, "22:00", "24:00", sub="불침번"),
    E("D07", "불침번", "duty", 9, "22:00", "24:00", sub="불침번"),
    # 교육·훈련 일정
    E("T01", "정훈교육", "training", 1, "14:00", "16:00"),
    E("T02", "사이버 보안 교육", "training", 2, "20:30", "21:30"),
    E("T03", "정신교육", "training", 3, "15:00", "17:00"),
    E("T04", "사격 훈련", "training", 4, "09:00", "12:00"),
    E("T05", "동계 피복 보급", "training", 4, "13:00", "17:00"),
    E("T06", "보안 점검 교육", "training", 7, "15:00", "17:00"),
    E("T07", "체력검정", "training", 8, "10:00", "12:00"),
    E("T08", "신체검사(의무실)", "training", 8, "10:30", "11:30"),
    E("T09", "독감 예방접종", "training", 9, "09:00", "10:30"),
    E("T10", "분대 전술훈련(오전)", "training", 10, "09:00", "12:00"),
    E("T11", "분대 전술훈련(오후)", "training", 10, "13:00", "17:30"),
    # 휴가 (금요일 오전부터 일요일 저녁까지)
    E("V01", "정기 휴가", "leave", 11, "09:00", "18:00", end_day=13),
]

# 근무 종류별 근무 후 최소 휴식 시간(시간) - 교육용 가상 기준
REST_RULE = {"불침번": 6, "당직": 12, "경계근무": 2}

# 개인 목표: priority 1(낮음)~5(높음), allow=하루 중 허용 시간대(분), pref=선호 시간대(분)
GOALS = [
    {"id": "G1", "name": "자격증 공부", "target": 720, "lens": [60, 90, 120], "day_cap": 180,
     "deadline": (10, "24:00"), "priority": 3, "allow": (540, 1260), "pref": (1110, 1260)},
    {"id": "G2", "name": "체력단련", "target": 300, "lens": [60, 90], "day_cap": 90,
     "deadline": (13, "24:00"), "priority": 2, "allow": (360, 1260), "pref": None},
    {"id": "G3", "name": "영어 단어 학습", "target": 240, "lens": [30, 60], "day_cap": 60,
     "deadline": (13, "24:00"), "priority": 1, "allow": (540, 1260), "pref": None},
]

# 일정 변경 공지 (가상): 원래 일정을 옮기거나 늘리거나 취소한다
CHANGES = [
    {"id": "S01", "from": "교육훈련과", "title": "사격 훈련 연기(11/06 -> 11/13)", "op": "move", "target": "T04",
     "day": 11, "start": "09:00", "end": "12:00"},
    {"id": "S02", "from": "교육훈련과", "title": "체력검정 일정 변경(11/10 -> 11/11)", "op": "move", "target": "T07",
     "day": 9, "start": "10:00", "end": "12:00"},
    {"id": "S03", "from": "교육훈련과", "title": "분대 전술훈련 오후 일정 야간 연장", "op": "extend", "target": "T11", "end": "21:00"},
    {"id": "S04", "from": "작전과", "title": "경계근무 취소", "op": "cancel", "target": "D03"},
]

KINDS = {"routine": "부대 일과", "duty": "근무", "training": "교육·훈련", "leave": "휴가"}


def validate(ev):
    """일정 한 건의 형식 오류 목록을 돌려준다(빈 목록이면 정상)."""
    errs = [f"{k} 누락" for k in ("id", "name", "kind", "day", "start", "end") if ev.get(k) in (None, "")]
    if ev.get("kind") not in KINDS:
        errs.append("종류 오류")
    for k in ("start", "end"):
        if ev.get(k) and not re.match(r"^\d{1,2}:[0-5]\d$", str(ev[k])):
            errs.append(f"{k} 시간 형식 오류")
    if not errs:
        if not 0 <= ev["day"] < 14:
            errs.append("기간(0~13일) 밖")
        elif t(ev.get("end_day", ev["day"]), ev["end"]) <= t(ev["day"], ev["start"]):
            errs.append("종료가 시작보다 빠르거나 같음")
    return errs


def to_df(raw):
    """일정 목록을 '시작·종료 = 11/02 00:00부터의 분' 표로 바꾼다."""
    rows = []
    for ev in raw:
        s, e = t(ev["day"], ev["start"]), t(ev.get("end_day", ev["day"]), ev["end"])
        rows.append({"id": ev["id"], "name": ev["name"], "kind": ev["kind"], "sub": ev.get("sub", ""),
                     "start": s, "end": e, "dur": e - s})
    return pd.DataFrame(rows).sort_values(["start", "end"]).reset_index(drop=True)


def fmt(x):
    d, r = divmod(int(x), 1440)
    day = START + dt.timedelta(days=d)
    return f"{day.month:02d}/{day.day:02d}({DOW[d % 7]}) {r // 60:02d}:{r % 60:02d}"

df = to_df(EVENTS)
df["시간(h)"] = (df["dur"] / 60).round(1)
print(df.assign(시작=df["start"].map(fmt), 종료=df["end"].map(fmt))[["id", "name", "kind", "시작", "종료", "시간(h)"]]
      .query("kind != 'routine'").head(12).to_string(index=False))

print(f"\n전체 일정 {len(df)}건 (2주 = 14일)")
cnt = df.groupby("kind").agg(건수=("id", "count"), 합계시간=("dur", lambda x: round(x.sum() / 60, 1)))
print(cnt.rename(index=KINDS).to_string())

errors = {e["id"]: validate(e) for e in EVENTS}
print("\n형식 검사 오류:", sum(len(v) for v in errors.values()), "건")

# 오류 데이터로 검사기 시험
bad = [
    E("X1", "시간 거꾸로", "duty", 2, "10:00", "09:00"),
    E("X2", "종류 오류", "회의", 2, "10:00", "11:00"),
    {"id": "X3", "name": "시간 형식 오류", "kind": "training", "day": 3, "start": "9시", "end": "11:00"},
    E("X4", "기간 밖", "training", 20, "09:00", "10:00"),
    E("X5", "", "training", 3, "09:00", "10:00"),
]
for b in bad:
    print(f"오류 주입 {b['id']}:", validate(b))


---

## 🎯 미션 02. 일정 시간 겹침 찾기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **일정이 시간상 겹치는 쌍을 찾아 성격별로 나눕니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **시작 시각순으로 정렬해 겹치는 쌍을 찾고, 실제 충돌(근무·교육·휴가끼리), 식사·점호 침범, 일과 대체(교육이 일과를 대신함), 일과 열외(근무·휴가자)로 분류 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `겹침 목록과 구분별 건수, 실제 충돌 목록, 검증 결과` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 충돌 탐지 — 겹침을 찾아 성격별로 분류하고 단순 비교와 결과를 맞춰 보았는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(겹침 목록과 구분별 건수, 실제 충돌 목록, 검증 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""일정 겹침을 찾아 실제 충돌·식사 침범·일과 대체·일과 열외로 나누고, 단순 전수 비교와 결과가 같은지 확인한다."""
import re
import datetime as dt
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 2주(11/02~11/15) 일정 자료 (실제 부대·인원·규정과 무관)
# 시각은 "일차(0=11/02 월요일) + HH:MM"으로 표기하며, 24시를 넘기면 26:00처럼 씁니다.
# ---------------------------------------------------------------------------
START = dt.date(2026, 11, 2)
DOW = "월화수목금토일"
DAY_END = "21:00"          # 개인 시간 계획은 저녁 점호(21:00) 전까지만 다룬다


def t(day, hhmm):
    # [TODO : 학생 실습 2] 일정 시간 겹침 찾기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def E(eid, name, kind, day, s, e, end_day=None, sub=""):
    # [TODO : 학생 실습 2] 일정 시간 겹침 찾기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_routine():
    ev = []
    for d in range(14):
        wd = d % 7
        if wd < 5:
            items = [("기상·점호", "fixed", "06:30", "07:00"), ("조식", "fixed", "07:00", "08:00"),
                     ("청소·정비", "fixed", "08:00", "09:00"), ("오전 일과", "work", "09:00", "12:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("오후 일과", "work", "13:00", "17:30"),
                     ("석식", "fixed", "17:30", "18:30"), ("저녁 점호", "fixed", "21:00", "21:30")]
        else:
            items = [("기상·점호", "fixed", "07:30", "08:00"), ("조식", "fixed", "08:00", "09:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("석식", "fixed", "17:30", "18:30"),
                     ("저녁 점호", "fixed", "21:00", "21:30")]
            if wd == 5:
                items.insert(2, ("주말 청소·정비", "work", "09:00", "10:30"))
        for k, (nm, sub, s, e) in enumerate(items):
            ev.append(E(f"R{d:02d}{k}", nm, "routine", d, s, e, sub=sub))
    return ev


EVENTS = make_routine() + [
    # 개인 근무표 (가상): 불침번·경계근무·당직 보조
    E("D01", "불침번", "duty", 0, "26:00", "28:00", sub="불침번"),
    E("D02", "불침번", "duty", 2, "22:00", "24:00", sub="불침번"),
    E("D03", "경계근무", "duty", 3, "16:00", "18:00", sub="경계근무"),
    E("D04", "불침번", "duty", 4, "26:00", "28:00", sub="불침번"),
    E("D05", "당직 보조", "duty", 6, "08:00", "32:00", sub="당직"),
    E("D06", "불침번", "duty", 8, "22:00", "24:00", sub="불침번"),
    E("D07", "불침번", "duty", 9, "22:00", "24:00", sub="불침번"),
    # 교육·훈련 일정
    E("T01", "정훈교육", "training", 1, "14:00", "16:00"),
    E("T02", "사이버 보안 교육", "training", 2, "20:30", "21:30"),
    E("T03", "정신교육", "training", 3, "15:00", "17:00"),
    E("T04", "사격 훈련", "training", 4, "09:00", "12:00"),
    E("T05", "동계 피복 보급", "training", 4, "13:00", "17:00"),
    E("T06", "보안 점검 교육", "training", 7, "15:00", "17:00"),
    E("T07", "체력검정", "training", 8, "10:00", "12:00"),
    E("T08", "신체검사(의무실)", "training", 8, "10:30", "11:30"),
    E("T09", "독감 예방접종", "training", 9, "09:00", "10:30"),
    E("T10", "분대 전술훈련(오전)", "training", 10, "09:00", "12:00"),
    E("T11", "분대 전술훈련(오후)", "training", 10, "13:00", "17:30"),
    # 휴가 (금요일 오전부터 일요일 저녁까지)
    E("V01", "정기 휴가", "leave", 11, "09:00", "18:00", end_day=13),
]

# 근무 종류별 근무 후 최소 휴식 시간(시간) - 교육용 가상 기준
REST_RULE = {"불침번": 6, "당직": 12, "경계근무": 2}

# 개인 목표: priority 1(낮음)~5(높음), allow=하루 중 허용 시간대(분), pref=선호 시간대(분)
GOALS = [
    {"id": "G1", "name": "자격증 공부", "target": 720, "lens": [60, 90, 120], "day_cap": 180,
     "deadline": (10, "24:00"), "priority": 3, "allow": (540, 1260), "pref": (1110, 1260)},
    {"id": "G2", "name": "체력단련", "target": 300, "lens": [60, 90], "day_cap": 90,
     "deadline": (13, "24:00"), "priority": 2, "allow": (360, 1260), "pref": None},
    {"id": "G3", "name": "영어 단어 학습", "target": 240, "lens": [30, 60], "day_cap": 60,
     "deadline": (13, "24:00"), "priority": 1, "allow": (540, 1260), "pref": None},
]

# 일정 변경 공지 (가상): 원래 일정을 옮기거나 늘리거나 취소한다
CHANGES = [
    {"id": "S01", "from": "교육훈련과", "title": "사격 훈련 연기(11/06 -> 11/13)", "op": "move", "target": "T04",
     "day": 11, "start": "09:00", "end": "12:00"},
    {"id": "S02", "from": "교육훈련과", "title": "체력검정 일정 변경(11/10 -> 11/11)", "op": "move", "target": "T07",
     "day": 9, "start": "10:00", "end": "12:00"},
    {"id": "S03", "from": "교육훈련과", "title": "분대 전술훈련 오후 일정 야간 연장", "op": "extend", "target": "T11", "end": "21:00"},
    {"id": "S04", "from": "작전과", "title": "경계근무 취소", "op": "cancel", "target": "D03"},
]

KINDS = {"routine": "부대 일과", "duty": "근무", "training": "교육·훈련", "leave": "휴가"}


def validate(ev):
    """일정 한 건의 형식 오류 목록을 돌려준다(빈 목록이면 정상)."""
    errs = [f"{k} 누락" for k in ("id", "name", "kind", "day", "start", "end") if ev.get(k) in (None, "")]
    if ev.get("kind") not in KINDS:
        errs.append("종류 오류")
    for k in ("start", "end"):
        if ev.get(k) and not re.match(r"^\d{1,2}:[0-5]\d$", str(ev[k])):
            errs.append(f"{k} 시간 형식 오류")
    if not errs:
        if not 0 <= ev["day"] < 14:
            errs.append("기간(0~13일) 밖")
        elif t(ev.get("end_day", ev["day"]), ev["end"]) <= t(ev["day"], ev["start"]):
            errs.append("종료가 시작보다 빠르거나 같음")
    return errs


def to_df(raw):
    """일정 목록을 '시작·종료 = 11/02 00:00부터의 분' 표로 바꾼다."""
    rows = []
    for ev in raw:
        s, e = t(ev["day"], ev["start"]), t(ev.get("end_day", ev["day"]), ev["end"])
        rows.append({"id": ev["id"], "name": ev["name"], "kind": ev["kind"], "sub": ev.get("sub", ""),
                     "start": s, "end": e, "dur": e - s})
    return pd.DataFrame(rows).sort_values(["start", "end"]).reset_index(drop=True)


def fmt(x):
    d, r = divmod(int(x), 1440)
    day = START + dt.timedelta(days=d)
    return f"{day.month:02d}/{day.day:02d}({DOW[d % 7]}) {r // 60:02d}:{r % 60:02d}"

def classify(a, b):
    """겹친 두 일정의 성격: hard(실제 충돌) / soft(식사·점호 침범) / replace(일과 대체) / exempt(근무·휴가자 일과 열외)"""
    if a["kind"] != "routine" and b["kind"] != "routine":
        return "hard"
    r, o = (a, b) if a["kind"] == "routine" else (b, a)
    if o["kind"] == "routine":
        return "hard"
    if o["kind"] in ("duty", "leave"):
        return "exempt"
    return "replace" if r["sub"] == "work" else "soft"


def find_conflicts(df):
    """시작 시각순으로 정렬된 일정에서 겹치는 쌍을 찾는다(끝난 일정 뒤로는 비교하지 않음)."""
    rows = df.to_dict("records")
    out = []
    for i, a in enumerate(rows):
        for b in rows[i + 1:]:
            if b["start"] >= a["end"]:
                break
            out.append({"일정A": f"{a['id']} {a['name']}", "일정B": f"{b['id']} {b['name']}",
                        "겹침(분)": min(a["end"], b["end"]) - b["start"], "시작": b["start"],
                        "구분": classify(a, b)})
    return pd.DataFrame(out, columns=["일정A", "일정B", "겹침(분)", "시작", "구분"])

df = to_df(EVENTS)
conf = find_conflicts(df)
print("겹침 구분별 건수:", conf["구분"].value_counts().to_dict())

hard = conf[conf["구분"] == "hard"]
print("\n[실제 충돌 - 사람이 조정해야 함]")
for r in hard.to_dict("records"):
    print(f"- {r['일정A']} / {r['일정B']}: {r['겹침(분)']}분 겹침 ({fmt(r['시작'])} 부근)")
soft = conf[conf["구분"] == "soft"]
print("\n[식사·점호 침범 - 확인 필요]")
for r in soft.to_dict("records"):
    print(f"- {r['일정A']} / {r['일정B']}: {r['겹침(분)']}분 ({fmt(r['시작'])})")

# 같은 결과가 나오는지 전수 비교(모든 쌍 검사)로 확인
rows = df.to_dict("records")
brute = sum(1 for i in range(len(rows)) for j in range(i + 1, len(rows))
            if rows[i]["start"] < rows[j]["end"] and rows[j]["start"] < rows[i]["end"])
print(f"\n정렬 방식 {len(conf)}쌍, 전수 비교 {brute}쌍 -> {'일치' if brute == len(conf) else '불일치'}")
print("목표 달성" if brute == len(conf) and len(hard) == 2 else "미달")

# 시험: 서로 붙어 있는 일정(끝=시작)은 겹침이 아니다
test = to_df([E("A", "가", "training", 1, "09:00", "10:00"), E("B", "나", "training", 1, "10:00", "11:00"),
              E("C", "다", "duty", 1, "10:30", "11:30")])
print("경계 시험:", find_conflicts(test)[["일정A", "일정B", "겹침(분)"]].values.tolist())


---

## 🎯 미션 03. 근무 후 휴식 시간 점검하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **근무가 끝난 뒤 다음 일과까지 휴식 시간이 기준을 지키는지 점검합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **근무 종류별 최소 휴식 기준(불침번 6시간, 당직 12시간, 경계근무 2시간)을 규칙표로 두고, 근무 종료부터 다음 일과·교육·근무 시작까지의 시간을 계산 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `휴식 점검표, 위반 목록과 조정 제안` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 규칙 점검 — 근무 후 휴식 규칙을 코드로 만들어 위반을 찾고 사람이 판단할 제안으로 정리했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(휴식 점검표, 위반 목록과 조정 제안)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""근무 종료 후 다음 일과까지의 휴식 시간이 기준(불침번 6시간·당직 12시간·경계근무 2시간)에 맞는지 점검하고 조정안을 제안한다."""
import re
import datetime as dt
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 2주(11/02~11/15) 일정 자료 (실제 부대·인원·규정과 무관)
# 시각은 "일차(0=11/02 월요일) + HH:MM"으로 표기하며, 24시를 넘기면 26:00처럼 씁니다.
# ---------------------------------------------------------------------------
START = dt.date(2026, 11, 2)
DOW = "월화수목금토일"
DAY_END = "21:00"          # 개인 시간 계획은 저녁 점호(21:00) 전까지만 다룬다


def t(day, hhmm):
    # [TODO : 학생 실습 3] 근무 후 휴식 시간 점검하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def E(eid, name, kind, day, s, e, end_day=None, sub=""):
    # [TODO : 학생 실습 3] 근무 후 휴식 시간 점검하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_routine():
    ev = []
    for d in range(14):
        wd = d % 7
        if wd < 5:
            items = [("기상·점호", "fixed", "06:30", "07:00"), ("조식", "fixed", "07:00", "08:00"),
                     ("청소·정비", "fixed", "08:00", "09:00"), ("오전 일과", "work", "09:00", "12:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("오후 일과", "work", "13:00", "17:30"),
                     ("석식", "fixed", "17:30", "18:30"), ("저녁 점호", "fixed", "21:00", "21:30")]
        else:
            items = [("기상·점호", "fixed", "07:30", "08:00"), ("조식", "fixed", "08:00", "09:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("석식", "fixed", "17:30", "18:30"),
                     ("저녁 점호", "fixed", "21:00", "21:30")]
            if wd == 5:
                items.insert(2, ("주말 청소·정비", "work", "09:00", "10:30"))
        for k, (nm, sub, s, e) in enumerate(items):
            ev.append(E(f"R{d:02d}{k}", nm, "routine", d, s, e, sub=sub))
    return ev


EVENTS = make_routine() + [
    # 개인 근무표 (가상): 불침번·경계근무·당직 보조
    E("D01", "불침번", "duty", 0, "26:00", "28:00", sub="불침번"),
    E("D02", "불침번", "duty", 2, "22:00", "24:00", sub="불침번"),
    E("D03", "경계근무", "duty", 3, "16:00", "18:00", sub="경계근무"),
    E("D04", "불침번", "duty", 4, "26:00", "28:00", sub="불침번"),
    E("D05", "당직 보조", "duty", 6, "08:00", "32:00", sub="당직"),
    E("D06", "불침번", "duty", 8, "22:00", "24:00", sub="불침번"),
    E("D07", "불침번", "duty", 9, "22:00", "24:00", sub="불침번"),
    # 교육·훈련 일정
    E("T01", "정훈교육", "training", 1, "14:00", "16:00"),
    E("T02", "사이버 보안 교육", "training", 2, "20:30", "21:30"),
    E("T03", "정신교육", "training", 3, "15:00", "17:00"),
    E("T04", "사격 훈련", "training", 4, "09:00", "12:00"),
    E("T05", "동계 피복 보급", "training", 4, "13:00", "17:00"),
    E("T06", "보안 점검 교육", "training", 7, "15:00", "17:00"),
    E("T07", "체력검정", "training", 8, "10:00", "12:00"),
    E("T08", "신체검사(의무실)", "training", 8, "10:30", "11:30"),
    E("T09", "독감 예방접종", "training", 9, "09:00", "10:30"),
    E("T10", "분대 전술훈련(오전)", "training", 10, "09:00", "12:00"),
    E("T11", "분대 전술훈련(오후)", "training", 10, "13:00", "17:30"),
    # 휴가 (금요일 오전부터 일요일 저녁까지)
    E("V01", "정기 휴가", "leave", 11, "09:00", "18:00", end_day=13),
]

# 근무 종류별 근무 후 최소 휴식 시간(시간) - 교육용 가상 기준
REST_RULE = {"불침번": 6, "당직": 12, "경계근무": 2}

# 개인 목표: priority 1(낮음)~5(높음), allow=하루 중 허용 시간대(분), pref=선호 시간대(분)
GOALS = [
    {"id": "G1", "name": "자격증 공부", "target": 720, "lens": [60, 90, 120], "day_cap": 180,
     "deadline": (10, "24:00"), "priority": 3, "allow": (540, 1260), "pref": (1110, 1260)},
    {"id": "G2", "name": "체력단련", "target": 300, "lens": [60, 90], "day_cap": 90,
     "deadline": (13, "24:00"), "priority": 2, "allow": (360, 1260), "pref": None},
    {"id": "G3", "name": "영어 단어 학습", "target": 240, "lens": [30, 60], "day_cap": 60,
     "deadline": (13, "24:00"), "priority": 1, "allow": (540, 1260), "pref": None},
]

# 일정 변경 공지 (가상): 원래 일정을 옮기거나 늘리거나 취소한다
CHANGES = [
    {"id": "S01", "from": "교육훈련과", "title": "사격 훈련 연기(11/06 -> 11/13)", "op": "move", "target": "T04",
     "day": 11, "start": "09:00", "end": "12:00"},
    {"id": "S02", "from": "교육훈련과", "title": "체력검정 일정 변경(11/10 -> 11/11)", "op": "move", "target": "T07",
     "day": 9, "start": "10:00", "end": "12:00"},
    {"id": "S03", "from": "교육훈련과", "title": "분대 전술훈련 오후 일정 야간 연장", "op": "extend", "target": "T11", "end": "21:00"},
    {"id": "S04", "from": "작전과", "title": "경계근무 취소", "op": "cancel", "target": "D03"},
]

KINDS = {"routine": "부대 일과", "duty": "근무", "training": "교육·훈련", "leave": "휴가"}


def validate(ev):
    """일정 한 건의 형식 오류 목록을 돌려준다(빈 목록이면 정상)."""
    errs = [f"{k} 누락" for k in ("id", "name", "kind", "day", "start", "end") if ev.get(k) in (None, "")]
    if ev.get("kind") not in KINDS:
        errs.append("종류 오류")
    for k in ("start", "end"):
        if ev.get(k) and not re.match(r"^\d{1,2}:[0-5]\d$", str(ev[k])):
            errs.append(f"{k} 시간 형식 오류")
    if not errs:
        if not 0 <= ev["day"] < 14:
            errs.append("기간(0~13일) 밖")
        elif t(ev.get("end_day", ev["day"]), ev["end"]) <= t(ev["day"], ev["start"]):
            errs.append("종료가 시작보다 빠르거나 같음")
    return errs


def to_df(raw):
    """일정 목록을 '시작·종료 = 11/02 00:00부터의 분' 표로 바꾼다."""
    rows = []
    for ev in raw:
        s, e = t(ev["day"], ev["start"]), t(ev.get("end_day", ev["day"]), ev["end"])
        rows.append({"id": ev["id"], "name": ev["name"], "kind": ev["kind"], "sub": ev.get("sub", ""),
                     "start": s, "end": e, "dur": e - s})
    return pd.DataFrame(rows).sort_values(["start", "end"]).reset_index(drop=True)


def fmt(x):
    d, r = divmod(int(x), 1440)
    day = START + dt.timedelta(days=d)
    return f"{day.month:02d}/{day.day:02d}({DOW[d % 7]}) {r // 60:02d}:{r % 60:02d}"

def check_rest(df):
    """근무가 끝난 뒤 다음 일과·교육·근무가 시작될 때까지의 휴식 시간이 기준 이상인지 점검한다."""
    work = df[(df["kind"].isin(["duty", "training"])) | ((df["kind"] == "routine") & (df["sub"] == "work"))]
    rows = []
    for d in df[df["kind"] == "duty"].itertuples():
        need = REST_RULE.get(d.sub, 0) * 60
        nxt = work[work["start"] >= d.end]
        if nxt.empty:
            continue
        n = nxt.iloc[0]
        rest = int(n["start"] - d.end)
        rows.append({"근무": f"{d.id} {d.name}", "종료": fmt(d.end), "다음 일정": n["name"], "다음 시작": fmt(n["start"]),
                     "휴식(시간)": round(rest / 60, 1), "기준(시간)": need / 60,
                     "판정": "적정" if rest >= need else "위반", "부족(분)": max(0, need - rest), "id": d.id})
    return pd.DataFrame(rows)

df = to_df(EVENTS)
rest = check_rest(df)
print(rest.drop(columns=["id"]).to_string(index=False))
bad = rest[rest["판정"] == "위반"]
print(f"\n근무 {len(rest)}건 중 휴식 위반 {len(bad)}건")

print("\n[조정안 제안 - 확정은 분대장·행정반]")
for r in bad.to_dict("records"):
    m = r["부족(분)"]
    print(f"- {r['근무']}: 다음 일정({r['다음 일정']})을 {m // 60}시간 {m % 60}분 늦추거나 근무 시간대를 옮기는 방안 검토")

# 규칙 시험: 근무 직후 일과가 이어지는 가상 사례
test = to_df([E("D", "불침번", "duty", 1, "26:00", "28:00", sub="불침번"),
              E("W", "오전 일과", "routine", 2, "09:00", "12:00", sub="work")])
print("\n시험(02:00~04:00 불침번 후 09:00 일과):", check_rest(test)[["휴식(시간)", "기준(시간)", "판정"]].values.tolist())


---

## 🎯 미션 04. 빈 시간 찾기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **모든 일정과 근무 후 휴식 구간을 뺀 날짜별 빈 시간을 구합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **하루 활동 시간(06:30~21:00)에서 일정을 합친 구간과 휴식 보호 구간을 빼 빈 시간 구간을 구하고 날짜별 표로 제작하고 안정성을 확보하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `날짜별 빈 시간 표, 보호 구간 비교, 검산 결과` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [선택] 빈 시간 계산 — 일정을 뺀 빈 시간을 구하고 휴식 보호 전후를 비교해 검산했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(날짜별 빈 시간 표, 보호 구간 비교, 검산 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""모든 일정과 근무 후 휴식 구간을 뺀 빈 시간을 구하고 날짜별 사용 가능한 시간을 표로 만든다."""
import re
import datetime as dt
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 2주(11/02~11/15) 일정 자료 (실제 부대·인원·규정과 무관)
# 시각은 "일차(0=11/02 월요일) + HH:MM"으로 표기하며, 24시를 넘기면 26:00처럼 씁니다.
# ---------------------------------------------------------------------------
START = dt.date(2026, 11, 2)
DOW = "월화수목금토일"
DAY_END = "21:00"          # 개인 시간 계획은 저녁 점호(21:00) 전까지만 다룬다


def t(day, hhmm):
    # [TODO : 학생 실습 4] 빈 시간 찾기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def E(eid, name, kind, day, s, e, end_day=None, sub=""):
    # [TODO : 학생 실습 4] 빈 시간 찾기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_routine():
    ev = []
    for d in range(14):
        wd = d % 7
        if wd < 5:
            items = [("기상·점호", "fixed", "06:30", "07:00"), ("조식", "fixed", "07:00", "08:00"),
                     ("청소·정비", "fixed", "08:00", "09:00"), ("오전 일과", "work", "09:00", "12:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("오후 일과", "work", "13:00", "17:30"),
                     ("석식", "fixed", "17:30", "18:30"), ("저녁 점호", "fixed", "21:00", "21:30")]
        else:
            items = [("기상·점호", "fixed", "07:30", "08:00"), ("조식", "fixed", "08:00", "09:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("석식", "fixed", "17:30", "18:30"),
                     ("저녁 점호", "fixed", "21:00", "21:30")]
            if wd == 5:
                items.insert(2, ("주말 청소·정비", "work", "09:00", "10:30"))
        for k, (nm, sub, s, e) in enumerate(items):
            ev.append(E(f"R{d:02d}{k}", nm, "routine", d, s, e, sub=sub))
    return ev


EVENTS = make_routine() + [
    # 개인 근무표 (가상): 불침번·경계근무·당직 보조
    E("D01", "불침번", "duty", 0, "26:00", "28:00", sub="불침번"),
    E("D02", "불침번", "duty", 2, "22:00", "24:00", sub="불침번"),
    E("D03", "경계근무", "duty", 3, "16:00", "18:00", sub="경계근무"),
    E("D04", "불침번", "duty", 4, "26:00", "28:00", sub="불침번"),
    E("D05", "당직 보조", "duty", 6, "08:00", "32:00", sub="당직"),
    E("D06", "불침번", "duty", 8, "22:00", "24:00", sub="불침번"),
    E("D07", "불침번", "duty", 9, "22:00", "24:00", sub="불침번"),
    # 교육·훈련 일정
    E("T01", "정훈교육", "training", 1, "14:00", "16:00"),
    E("T02", "사이버 보안 교육", "training", 2, "20:30", "21:30"),
    E("T03", "정신교육", "training", 3, "15:00", "17:00"),
    E("T04", "사격 훈련", "training", 4, "09:00", "12:00"),
    E("T05", "동계 피복 보급", "training", 4, "13:00", "17:00"),
    E("T06", "보안 점검 교육", "training", 7, "15:00", "17:00"),
    E("T07", "체력검정", "training", 8, "10:00", "12:00"),
    E("T08", "신체검사(의무실)", "training", 8, "10:30", "11:30"),
    E("T09", "독감 예방접종", "training", 9, "09:00", "10:30"),
    E("T10", "분대 전술훈련(오전)", "training", 10, "09:00", "12:00"),
    E("T11", "분대 전술훈련(오후)", "training", 10, "13:00", "17:30"),
    # 휴가 (금요일 오전부터 일요일 저녁까지)
    E("V01", "정기 휴가", "leave", 11, "09:00", "18:00", end_day=13),
]

# 근무 종류별 근무 후 최소 휴식 시간(시간) - 교육용 가상 기준
REST_RULE = {"불침번": 6, "당직": 12, "경계근무": 2}

# 개인 목표: priority 1(낮음)~5(높음), allow=하루 중 허용 시간대(분), pref=선호 시간대(분)
GOALS = [
    {"id": "G1", "name": "자격증 공부", "target": 720, "lens": [60, 90, 120], "day_cap": 180,
     "deadline": (10, "24:00"), "priority": 3, "allow": (540, 1260), "pref": (1110, 1260)},
    {"id": "G2", "name": "체력단련", "target": 300, "lens": [60, 90], "day_cap": 90,
     "deadline": (13, "24:00"), "priority": 2, "allow": (360, 1260), "pref": None},
    {"id": "G3", "name": "영어 단어 학습", "target": 240, "lens": [30, 60], "day_cap": 60,
     "deadline": (13, "24:00"), "priority": 1, "allow": (540, 1260), "pref": None},
]

# 일정 변경 공지 (가상): 원래 일정을 옮기거나 늘리거나 취소한다
CHANGES = [
    {"id": "S01", "from": "교육훈련과", "title": "사격 훈련 연기(11/06 -> 11/13)", "op": "move", "target": "T04",
     "day": 11, "start": "09:00", "end": "12:00"},
    {"id": "S02", "from": "교육훈련과", "title": "체력검정 일정 변경(11/10 -> 11/11)", "op": "move", "target": "T07",
     "day": 9, "start": "10:00", "end": "12:00"},
    {"id": "S03", "from": "교육훈련과", "title": "분대 전술훈련 오후 일정 야간 연장", "op": "extend", "target": "T11", "end": "21:00"},
    {"id": "S04", "from": "작전과", "title": "경계근무 취소", "op": "cancel", "target": "D03"},
]

KINDS = {"routine": "부대 일과", "duty": "근무", "training": "교육·훈련", "leave": "휴가"}


def validate(ev):
    """일정 한 건의 형식 오류 목록을 돌려준다(빈 목록이면 정상)."""
    errs = [f"{k} 누락" for k in ("id", "name", "kind", "day", "start", "end") if ev.get(k) in (None, "")]
    if ev.get("kind") not in KINDS:
        errs.append("종류 오류")
    for k in ("start", "end"):
        if ev.get(k) and not re.match(r"^\d{1,2}:[0-5]\d$", str(ev[k])):
            errs.append(f"{k} 시간 형식 오류")
    if not errs:
        if not 0 <= ev["day"] < 14:
            errs.append("기간(0~13일) 밖")
        elif t(ev.get("end_day", ev["day"]), ev["end"]) <= t(ev["day"], ev["start"]):
            errs.append("종료가 시작보다 빠르거나 같음")
    return errs


def to_df(raw):
    """일정 목록을 '시작·종료 = 11/02 00:00부터의 분' 표로 바꾼다."""
    rows = []
    for ev in raw:
        s, e = t(ev["day"], ev["start"]), t(ev.get("end_day", ev["day"]), ev["end"])
        rows.append({"id": ev["id"], "name": ev["name"], "kind": ev["kind"], "sub": ev.get("sub", ""),
                     "start": s, "end": e, "dur": e - s})
    return pd.DataFrame(rows).sort_values(["start", "end"]).reset_index(drop=True)


def fmt(x):
    d, r = divmod(int(x), 1440)
    day = START + dt.timedelta(days=d)
    return f"{day.month:02d}/{day.day:02d}({DOW[d % 7]}) {r // 60:02d}:{r % 60:02d}"

def check_rest(df):
    """근무가 끝난 뒤 다음 일과·교육·근무가 시작될 때까지의 휴식 시간이 기준 이상인지 점검한다."""
    work = df[(df["kind"].isin(["duty", "training"])) | ((df["kind"] == "routine") & (df["sub"] == "work"))]
    rows = []
    for d in df[df["kind"] == "duty"].itertuples():
        need = REST_RULE.get(d.sub, 0) * 60
        nxt = work[work["start"] >= d.end]
        if nxt.empty:
            continue
        n = nxt.iloc[0]
        rest = int(n["start"] - d.end)
        rows.append({"근무": f"{d.id} {d.name}", "종료": fmt(d.end), "다음 일정": n["name"], "다음 시작": fmt(n["start"]),
                     "휴식(시간)": round(rest / 60, 1), "기준(시간)": need / 60,
                     "판정": "적정" if rest >= need else "위반", "부족(분)": max(0, need - rest), "id": d.id})
    return pd.DataFrame(rows)

def day_window(d):
    return (t(d, "06:30") if d % 7 < 5 else t(d, "07:30")), t(d, DAY_END)


def merge(iv):
    out = []
    for s, e in sorted(iv):
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out


def free_intervals(df, min_len=30, protect=True):
    """일과·근무·교육·휴가를 모두 뺀 빈 시간 구간. protect=True면 근무 후 휴식 구간도 뺀다."""
    busy = [(r.start, r.end) for r in df.itertuples()]
    if protect:
        busy += [(r.end, r.end + REST_RULE.get(r.sub, 0) * 60) for r in df[df["kind"] == "duty"].itertuples()]
    busy = merge(busy)
    free = []
    for d in range(14):
        lo, hi = day_window(d)
        cur = lo
        for s, e in busy:
            if e <= cur or s >= hi:
                continue
            if s > cur:
                free.append((cur, min(s, hi)))
            cur = max(cur, e)
        if cur < hi:
            free.append((cur, hi))
    return [(s, e) for s, e in free if e - s >= min_len]

df = to_df(EVENTS)
free_raw = free_intervals(df, protect=False)
free = free_intervals(df, protect=True)

rows = []
for d in range(14):
    lo, hi = d * 1440, (d + 1) * 1440
    part = [(s, e) for s, e in free if lo <= s < hi]
    raw_min = sum(e - s for s, e in free_raw if lo <= s < hi)
    rows.append({"날짜": f"{START + dt.timedelta(days=d):%m/%d}({DOW[d % 7]})", "빈 구간": ", ".join(f"{fmt(s)[-5:]}~{fmt(e)[-5:]}" for s, e in part) or "-",
                 "빈 시간(분)": sum(e - s for s, e in part), "휴식 보호 전(분)": raw_min})
tab = pd.DataFrame(rows)
print(tab.to_string(index=False))

tot, tot_raw = tab["빈 시간(분)"].sum(), tab["휴식 보호 전(분)"].sum()
print(f"\n2주 빈 시간 합계 {tot}분({tot / 60:.1f}시간), 휴식 보호 전 {tot_raw}분 -> 보호 구간 {tot_raw - tot}분 제외")
print("빈 시간이 없는 날:", [r["날짜"] for r in rows if r["빈 시간(분)"] == 0])

# 검산: 빈 시간 + 일정 시간 = 하루 활동 시간(06:30~21:00, 주말 07:30~21:00)
chk = 0
for d in range(14):
    lo, hi = day_window(d)
    busy = merge([(max(r.start, lo), min(r.end, hi)) for r in df.itertuples() if r.end > lo and r.start < hi]
                 + [(max(r.end, lo), min(r.end + REST_RULE.get(r.sub, 0) * 60, hi)) for r in df[df["kind"] == "duty"].itertuples()
                    if r.end + REST_RULE.get(r.sub, 0) * 60 > lo and r.end < hi])
    chk += (hi - lo) - sum(e - s for s, e in busy)
print("검산(구간 합과 하루 길이 차):", chk, "분 ->", "일치" if chk == tot else "불일치")


---

## 🎯 미션 05. 개인 목표 시간 배정하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **개인 목표(자격증 공부·체력단련·영어 학습)를 빈 시간에 탐욕과 최적화로 배정하고 비교합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **목표마다 총 시간, 블록 길이, 하루 상한, 마감을 정하고 탐욕 방법(우선순위 높은 목표부터 이른 시간에 채움)으로 배정 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `탐욕·최적화 배정표, 두 방법 비교표` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 배정 알고리즘 — 탐욕과 최적화 두 방법으로 배정하고 결과를 수치로 비교했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(탐욕·최적화 배정표, 두 방법 비교표)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""개인 목표(자격증 공부·체력단련·영어 학습)를 빈 시간에 탐욕 방법과 최적화(scipy.optimize.milp)로 배정하고 결과를 비교한다."""
import re
import datetime as dt
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import coo_matrix

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 2주(11/02~11/15) 일정 자료 (실제 부대·인원·규정과 무관)
# 시각은 "일차(0=11/02 월요일) + HH:MM"으로 표기하며, 24시를 넘기면 26:00처럼 씁니다.
# ---------------------------------------------------------------------------
START = dt.date(2026, 11, 2)
DOW = "월화수목금토일"
DAY_END = "21:00"          # 개인 시간 계획은 저녁 점호(21:00) 전까지만 다룬다


def t(day, hhmm):
    # [TODO : 학생 실습 5] 개인 목표 시간 배정하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def E(eid, name, kind, day, s, e, end_day=None, sub=""):
    # [TODO : 학생 실습 5] 개인 목표 시간 배정하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_routine():
    ev = []
    for d in range(14):
        wd = d % 7
        if wd < 5:
            items = [("기상·점호", "fixed", "06:30", "07:00"), ("조식", "fixed", "07:00", "08:00"),
                     ("청소·정비", "fixed", "08:00", "09:00"), ("오전 일과", "work", "09:00", "12:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("오후 일과", "work", "13:00", "17:30"),
                     ("석식", "fixed", "17:30", "18:30"), ("저녁 점호", "fixed", "21:00", "21:30")]
        else:
            items = [("기상·점호", "fixed", "07:30", "08:00"), ("조식", "fixed", "08:00", "09:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("석식", "fixed", "17:30", "18:30"),
                     ("저녁 점호", "fixed", "21:00", "21:30")]
            if wd == 5:
                items.insert(2, ("주말 청소·정비", "work", "09:00", "10:30"))
        for k, (nm, sub, s, e) in enumerate(items):
            ev.append(E(f"R{d:02d}{k}", nm, "routine", d, s, e, sub=sub))
    return ev


EVENTS = make_routine() + [
    # 개인 근무표 (가상): 불침번·경계근무·당직 보조
    E("D01", "불침번", "duty", 0, "26:00", "28:00", sub="불침번"),
    E("D02", "불침번", "duty", 2, "22:00", "24:00", sub="불침번"),
    E("D03", "경계근무", "duty", 3, "16:00", "18:00", sub="경계근무"),
    E("D04", "불침번", "duty", 4, "26:00", "28:00", sub="불침번"),
    E("D05", "당직 보조", "duty", 6, "08:00", "32:00", sub="당직"),
    E("D06", "불침번", "duty", 8, "22:00", "24:00", sub="불침번"),
    E("D07", "불침번", "duty", 9, "22:00", "24:00", sub="불침번"),
    # 교육·훈련 일정
    E("T01", "정훈교육", "training", 1, "14:00", "16:00"),
    E("T02", "사이버 보안 교육", "training", 2, "20:30", "21:30"),
    E("T03", "정신교육", "training", 3, "15:00", "17:00"),
    E("T04", "사격 훈련", "training", 4, "09:00", "12:00"),
    E("T05", "동계 피복 보급", "training", 4, "13:00", "17:00"),
    E("T06", "보안 점검 교육", "training", 7, "15:00", "17:00"),
    E("T07", "체력검정", "training", 8, "10:00", "12:00"),
    E("T08", "신체검사(의무실)", "training", 8, "10:30", "11:30"),
    E("T09", "독감 예방접종", "training", 9, "09:00", "10:30"),
    E("T10", "분대 전술훈련(오전)", "training", 10, "09:00", "12:00"),
    E("T11", "분대 전술훈련(오후)", "training", 10, "13:00", "17:30"),
    # 휴가 (금요일 오전부터 일요일 저녁까지)
    E("V01", "정기 휴가", "leave", 11, "09:00", "18:00", end_day=13),
]

# 근무 종류별 근무 후 최소 휴식 시간(시간) - 교육용 가상 기준
REST_RULE = {"불침번": 6, "당직": 12, "경계근무": 2}

# 개인 목표: priority 1(낮음)~5(높음), allow=하루 중 허용 시간대(분), pref=선호 시간대(분)
GOALS = [
    {"id": "G1", "name": "자격증 공부", "target": 720, "lens": [60, 90, 120], "day_cap": 180,
     "deadline": (10, "24:00"), "priority": 3, "allow": (540, 1260), "pref": (1110, 1260)},
    {"id": "G2", "name": "체력단련", "target": 300, "lens": [60, 90], "day_cap": 90,
     "deadline": (13, "24:00"), "priority": 2, "allow": (360, 1260), "pref": None},
    {"id": "G3", "name": "영어 단어 학습", "target": 240, "lens": [30, 60], "day_cap": 60,
     "deadline": (13, "24:00"), "priority": 1, "allow": (540, 1260), "pref": None},
]

# 일정 변경 공지 (가상): 원래 일정을 옮기거나 늘리거나 취소한다
CHANGES = [
    {"id": "S01", "from": "교육훈련과", "title": "사격 훈련 연기(11/06 -> 11/13)", "op": "move", "target": "T04",
     "day": 11, "start": "09:00", "end": "12:00"},
    {"id": "S02", "from": "교육훈련과", "title": "체력검정 일정 변경(11/10 -> 11/11)", "op": "move", "target": "T07",
     "day": 9, "start": "10:00", "end": "12:00"},
    {"id": "S03", "from": "교육훈련과", "title": "분대 전술훈련 오후 일정 야간 연장", "op": "extend", "target": "T11", "end": "21:00"},
    {"id": "S04", "from": "작전과", "title": "경계근무 취소", "op": "cancel", "target": "D03"},
]

KINDS = {"routine": "부대 일과", "duty": "근무", "training": "교육·훈련", "leave": "휴가"}


def validate(ev):
    """일정 한 건의 형식 오류 목록을 돌려준다(빈 목록이면 정상)."""
    errs = [f"{k} 누락" for k in ("id", "name", "kind", "day", "start", "end") if ev.get(k) in (None, "")]
    if ev.get("kind") not in KINDS:
        errs.append("종류 오류")
    for k in ("start", "end"):
        if ev.get(k) and not re.match(r"^\d{1,2}:[0-5]\d$", str(ev[k])):
            errs.append(f"{k} 시간 형식 오류")
    if not errs:
        if not 0 <= ev["day"] < 14:
            errs.append("기간(0~13일) 밖")
        elif t(ev.get("end_day", ev["day"]), ev["end"]) <= t(ev["day"], ev["start"]):
            errs.append("종료가 시작보다 빠르거나 같음")
    return errs


def to_df(raw):
    """일정 목록을 '시작·종료 = 11/02 00:00부터의 분' 표로 바꾼다."""
    rows = []
    for ev in raw:
        s, e = t(ev["day"], ev["start"]), t(ev.get("end_day", ev["day"]), ev["end"])
        rows.append({"id": ev["id"], "name": ev["name"], "kind": ev["kind"], "sub": ev.get("sub", ""),
                     "start": s, "end": e, "dur": e - s})
    return pd.DataFrame(rows).sort_values(["start", "end"]).reset_index(drop=True)


def fmt(x):
    d, r = divmod(int(x), 1440)
    day = START + dt.timedelta(days=d)
    return f"{day.month:02d}/{day.day:02d}({DOW[d % 7]}) {r // 60:02d}:{r % 60:02d}"

def check_rest(df):
    """근무가 끝난 뒤 다음 일과·교육·근무가 시작될 때까지의 휴식 시간이 기준 이상인지 점검한다."""
    work = df[(df["kind"].isin(["duty", "training"])) | ((df["kind"] == "routine") & (df["sub"] == "work"))]
    rows = []
    for d in df[df["kind"] == "duty"].itertuples():
        need = REST_RULE.get(d.sub, 0) * 60
        nxt = work[work["start"] >= d.end]
        if nxt.empty:
            continue
        n = nxt.iloc[0]
        rest = int(n["start"] - d.end)
        rows.append({"근무": f"{d.id} {d.name}", "종료": fmt(d.end), "다음 일정": n["name"], "다음 시작": fmt(n["start"]),
                     "휴식(시간)": round(rest / 60, 1), "기준(시간)": need / 60,
                     "판정": "적정" if rest >= need else "위반", "부족(분)": max(0, need - rest), "id": d.id})
    return pd.DataFrame(rows)

def day_window(d):
    return (t(d, "06:30") if d % 7 < 5 else t(d, "07:30")), t(d, DAY_END)


def merge(iv):
    out = []
    for s, e in sorted(iv):
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out


def free_intervals(df, min_len=30, protect=True):
    """일과·근무·교육·휴가를 모두 뺀 빈 시간 구간. protect=True면 근무 후 휴식 구간도 뺀다."""
    busy = [(r.start, r.end) for r in df.itertuples()]
    if protect:
        busy += [(r.end, r.end + REST_RULE.get(r.sub, 0) * 60) for r in df[df["kind"] == "duty"].itertuples()]
    busy = merge(busy)
    free = []
    for d in range(14):
        lo, hi = day_window(d)
        cur = lo
        for s, e in busy:
            if e <= cur or s >= hi:
                continue
            if s > cur:
                free.append((cur, min(s, hi)))
            cur = max(cur, e)
        if cur < hi:
            free.append((cur, hi))
    return [(s, e) for s, e in free if e - s >= min_len]

SLOT = 30                                   # 계획은 30분 칸 단위
PW = {1: 1.0, 2: 2.0, 3: 3.0, 4: 4.0, 5: 5.0}   # 우선순위 가중치


def cand_blocks(free, g):
    """목표 g가 들어갈 수 있는 모든 후보 블록(시작, 종료)."""
    dl = t(*g["deadline"])
    out = []
    for s, e in free:
        d = s // 1440
        a, b = max(s, d * 1440 + g["allow"][0]), min(e, d * 1440 + g["allow"][1], dl)
        for st in range(-(-a // SLOT) * SLOT, b, SLOT):
            for L in g["lens"]:
                if st + L <= b:
                    out.append((g["id"], st, st + L))
    return out


def block_value(g, st, en, mode="priority"):
    L = en - st
    v = (PW[g["priority"]] if mode == "priority" else 1.0) * L
    if g["pref"] and g["pref"][0] <= st % 1440 and (en - 1) % 1440 < g["pref"][1]:
        v += 0.2 * L                        # 선호 시간대(저녁) 가산
    return v - 5 - 0.0005 * st              # 블록 수는 적게, 같은 값이면 이른 시각 우선


def plan_greedy(free, goals, day_total_cap=240):
    """탐욕: 우선순위 높은 목표부터, 이른 빈 시간부터 차례로 채운다."""
    ok = set()
    for s, e in free:
        ok.update(range(s // SLOT, e // SLOT))
    order = sorted(goals, key=lambda g: (-g["priority"], t(*g["deadline"])))
    used, blocks = {}, []
    for g in order:
        remain, dl = g["target"], t(*g["deadline"]) // SLOT
        lo, hi, mn, mx = g["allow"][0] // SLOT, g["allow"][1] // SLOT, min(g["lens"]), max(g["lens"])
        for d in range(14):
            cap = min(g["day_cap"], day_total_cap - used.get(d, 0), remain)
            k, end = d * 48 + lo, min(d * 48 + hi, dl)
            while remain > 0 and cap >= mn and k < end:
                if k not in ok:
                    k += 1
                    continue
                run = 0
                while k + run < end and (k + run) in ok:
                    run += 1
                L = min(mx, run * SLOT, cap) // SLOT * SLOT
                if L >= mn:
                    blocks.append((g["id"], k * SLOT, k * SLOT + L))
                    ok.difference_update(range(k, k + L // SLOT))
                    cap -= L; remain -= L; used[d] = used.get(d, 0) + L
                    k += L // SLOT
                else:
                    k += run
    return sorted(blocks, key=lambda b: b[1])


def plan_milp(free, goals, mode="priority", day_total_cap=240, prev=None, bonus=0.0):
    """최적화: 후보 블록 중 겹치지 않게 고르되 목표 시간·하루 상한·마감을 지키며 가치 합을 최대로 한다."""
    G = {g["id"]: g for g in goals}
    cands = [c for g in goals for c in cand_blocks(free, g)]
    if not cands:
        return []
    groups = {}
    for i, (gid, s, e) in enumerate(cands):
        for k in range(s // SLOT, e // SLOT):
            groups.setdefault(("slot", k), (1, []))[1].append((i, 1))
        groups.setdefault(("goal", gid), (G[gid]["target"], []))[1].append((i, e - s))
        groups.setdefault(("gday", gid, s // 1440), (G[gid]["day_cap"], []))[1].append((i, e - s))
        groups.setdefault(("day", s // 1440), (day_total_cap, []))[1].append((i, e - s))
    r, c, v, ub = [], [], [], []
    for row, (cap, items) in enumerate(groups.values()):
        ub.append(cap)
        for i, coef in items:
            r.append(row); c.append(i); v.append(coef)
    A = coo_matrix((v, (r, c)), shape=(len(ub), len(cands)))
    val = np.array([block_value(G[g], s, e, mode) + (bonus * (e - s) if prev and (g, s, e) in prev else 0)
                    for g, s, e in cands])
    res = milp(-val, constraints=LinearConstraint(A, -np.inf, np.array(ub, float)),
               integrality=np.ones(len(cands)), bounds=Bounds(0, 1), options={"time_limit": 20})
    if res.x is None:
        return []
    return sorted([cands[i] for i in range(len(cands)) if res.x[i] > 0.5], key=lambda b: b[1])


def evaluate(blocks, goals):
    rows = []
    for g in goals:
        bs = [b for b in blocks if b[0] == g["id"]]
        got = sum(e - s for _, s, e in bs)
        late = sum(1 for _, s, e in bs if e > t(*g["deadline"]))
        rows.append({"목표": g["name"], "요구(분)": g["target"], "배정(분)": got, "달성률": f"{got / g['target']:.0%}",
                     "블록 수": len(bs), "마감 초과": late,
                     "판정": "목표 달성" if got >= g["target"] and late == 0 else "미달"})
    return pd.DataFrame(rows)


def blocks_outside_free(blocks, free):
    """빈 시간 밖(다른 일정이나 휴식 보호 구간과 겹침)에 놓인 블록 수"""
    return sum(1 for _, s, e in blocks if not any(fs <= s and e <= fe for fs, fe in free))

df = to_df(EVENTS)
free = free_intervals(df)
names = {g["id"]: g["name"] for g in GOALS}

greedy = plan_greedy(free, GOALS)
opt = plan_milp(free, GOALS)
for title, bl in (("탐욕(우선순위 높은 목표부터 이른 시간에)", greedy), ("최적화(milp)", opt)):
    print(f"[{title}]")
    print(evaluate(bl, GOALS).to_string(index=False))
    per_day = pd.Series([e - s for _, s, e in bl], index=[s // 1440 for _, s, e in bl]).groupby(level=0).sum()
    pref = sum(e - s for g, s, e in bl if g == "G1" and 1110 <= s % 1440 and (e - 1) % 1440 < 1260)
    print(f"  블록 {len(bl)}개, 하루 최대 {per_day.max()}분, 사용한 날 {per_day.size}일, 자격증 공부 저녁 선호 시간 {pref}분")
    print(f"  빈 시간 밖에 놓인 블록(충돌) {blocks_outside_free(bl, free)}개\n")

print("[최적화 배정 일부]")
for g, s, e in opt[:8]:
    print(f"- {fmt(s)}~{fmt(e)[-5:]} {names[g]} ({e - s}분)")
print("\n목표 달성" if all(evaluate(opt, GOALS)["판정"] == "목표 달성") and blocks_outside_free(opt, free) == 0 else "\n미달")


---

## 🎯 미션 06. 우선순위·마감 반영하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **우선순위와 마감을 최적화에 반영해 시간이 모자랄 때의 동작을 확인합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **하루 개인 시간 상한을 줄여 시간을 모자라게 만든 뒤 우선순위를 무시한 경우와 반영한 경우를 비교 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `우선순위 비교표, 마감 시험 결과, 부족분 제안` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 우선순위·마감 반영 — 우선순위와 마감이 결과에 미치는 영향을 수치로 확인하고 부족분 대응을 제안했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(우선순위 비교표, 마감 시험 결과, 부족분 제안)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""우선순위 가중치와 마감을 최적화에 반영해, 시간이 모자랄 때 중요한 목표와 마감이 가까운 목표가 먼저 채워지는지 확인한다."""
import re
import datetime as dt
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import coo_matrix

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 2주(11/02~11/15) 일정 자료 (실제 부대·인원·규정과 무관)
# 시각은 "일차(0=11/02 월요일) + HH:MM"으로 표기하며, 24시를 넘기면 26:00처럼 씁니다.
# ---------------------------------------------------------------------------
START = dt.date(2026, 11, 2)
DOW = "월화수목금토일"
DAY_END = "21:00"          # 개인 시간 계획은 저녁 점호(21:00) 전까지만 다룬다


def t(day, hhmm):
    # [TODO : 학생 실습 6] 우선순위·마감 반영하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def E(eid, name, kind, day, s, e, end_day=None, sub=""):
    # [TODO : 학생 실습 6] 우선순위·마감 반영하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_routine():
    ev = []
    for d in range(14):
        wd = d % 7
        if wd < 5:
            items = [("기상·점호", "fixed", "06:30", "07:00"), ("조식", "fixed", "07:00", "08:00"),
                     ("청소·정비", "fixed", "08:00", "09:00"), ("오전 일과", "work", "09:00", "12:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("오후 일과", "work", "13:00", "17:30"),
                     ("석식", "fixed", "17:30", "18:30"), ("저녁 점호", "fixed", "21:00", "21:30")]
        else:
            items = [("기상·점호", "fixed", "07:30", "08:00"), ("조식", "fixed", "08:00", "09:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("석식", "fixed", "17:30", "18:30"),
                     ("저녁 점호", "fixed", "21:00", "21:30")]
            if wd == 5:
                items.insert(2, ("주말 청소·정비", "work", "09:00", "10:30"))
        for k, (nm, sub, s, e) in enumerate(items):
            ev.append(E(f"R{d:02d}{k}", nm, "routine", d, s, e, sub=sub))
    return ev


EVENTS = make_routine() + [
    # 개인 근무표 (가상): 불침번·경계근무·당직 보조
    E("D01", "불침번", "duty", 0, "26:00", "28:00", sub="불침번"),
    E("D02", "불침번", "duty", 2, "22:00", "24:00", sub="불침번"),
    E("D03", "경계근무", "duty", 3, "16:00", "18:00", sub="경계근무"),
    E("D04", "불침번", "duty", 4, "26:00", "28:00", sub="불침번"),
    E("D05", "당직 보조", "duty", 6, "08:00", "32:00", sub="당직"),
    E("D06", "불침번", "duty", 8, "22:00", "24:00", sub="불침번"),
    E("D07", "불침번", "duty", 9, "22:00", "24:00", sub="불침번"),
    # 교육·훈련 일정
    E("T01", "정훈교육", "training", 1, "14:00", "16:00"),
    E("T02", "사이버 보안 교육", "training", 2, "20:30", "21:30"),
    E("T03", "정신교육", "training", 3, "15:00", "17:00"),
    E("T04", "사격 훈련", "training", 4, "09:00", "12:00"),
    E("T05", "동계 피복 보급", "training", 4, "13:00", "17:00"),
    E("T06", "보안 점검 교육", "training", 7, "15:00", "17:00"),
    E("T07", "체력검정", "training", 8, "10:00", "12:00"),
    E("T08", "신체검사(의무실)", "training", 8, "10:30", "11:30"),
    E("T09", "독감 예방접종", "training", 9, "09:00", "10:30"),
    E("T10", "분대 전술훈련(오전)", "training", 10, "09:00", "12:00"),
    E("T11", "분대 전술훈련(오후)", "training", 10, "13:00", "17:30"),
    # 휴가 (금요일 오전부터 일요일 저녁까지)
    E("V01", "정기 휴가", "leave", 11, "09:00", "18:00", end_day=13),
]

# 근무 종류별 근무 후 최소 휴식 시간(시간) - 교육용 가상 기준
REST_RULE = {"불침번": 6, "당직": 12, "경계근무": 2}

# 개인 목표: priority 1(낮음)~5(높음), allow=하루 중 허용 시간대(분), pref=선호 시간대(분)
GOALS = [
    {"id": "G1", "name": "자격증 공부", "target": 720, "lens": [60, 90, 120], "day_cap": 180,
     "deadline": (10, "24:00"), "priority": 3, "allow": (540, 1260), "pref": (1110, 1260)},
    {"id": "G2", "name": "체력단련", "target": 300, "lens": [60, 90], "day_cap": 90,
     "deadline": (13, "24:00"), "priority": 2, "allow": (360, 1260), "pref": None},
    {"id": "G3", "name": "영어 단어 학습", "target": 240, "lens": [30, 60], "day_cap": 60,
     "deadline": (13, "24:00"), "priority": 1, "allow": (540, 1260), "pref": None},
]

# 일정 변경 공지 (가상): 원래 일정을 옮기거나 늘리거나 취소한다
CHANGES = [
    {"id": "S01", "from": "교육훈련과", "title": "사격 훈련 연기(11/06 -> 11/13)", "op": "move", "target": "T04",
     "day": 11, "start": "09:00", "end": "12:00"},
    {"id": "S02", "from": "교육훈련과", "title": "체력검정 일정 변경(11/10 -> 11/11)", "op": "move", "target": "T07",
     "day": 9, "start": "10:00", "end": "12:00"},
    {"id": "S03", "from": "교육훈련과", "title": "분대 전술훈련 오후 일정 야간 연장", "op": "extend", "target": "T11", "end": "21:00"},
    {"id": "S04", "from": "작전과", "title": "경계근무 취소", "op": "cancel", "target": "D03"},
]

KINDS = {"routine": "부대 일과", "duty": "근무", "training": "교육·훈련", "leave": "휴가"}


def validate(ev):
    """일정 한 건의 형식 오류 목록을 돌려준다(빈 목록이면 정상)."""
    errs = [f"{k} 누락" for k in ("id", "name", "kind", "day", "start", "end") if ev.get(k) in (None, "")]
    if ev.get("kind") not in KINDS:
        errs.append("종류 오류")
    for k in ("start", "end"):
        if ev.get(k) and not re.match(r"^\d{1,2}:[0-5]\d$", str(ev[k])):
            errs.append(f"{k} 시간 형식 오류")
    if not errs:
        if not 0 <= ev["day"] < 14:
            errs.append("기간(0~13일) 밖")
        elif t(ev.get("end_day", ev["day"]), ev["end"]) <= t(ev["day"], ev["start"]):
            errs.append("종료가 시작보다 빠르거나 같음")
    return errs


def to_df(raw):
    """일정 목록을 '시작·종료 = 11/02 00:00부터의 분' 표로 바꾼다."""
    rows = []
    for ev in raw:
        s, e = t(ev["day"], ev["start"]), t(ev.get("end_day", ev["day"]), ev["end"])
        rows.append({"id": ev["id"], "name": ev["name"], "kind": ev["kind"], "sub": ev.get("sub", ""),
                     "start": s, "end": e, "dur": e - s})
    return pd.DataFrame(rows).sort_values(["start", "end"]).reset_index(drop=True)


def fmt(x):
    d, r = divmod(int(x), 1440)
    day = START + dt.timedelta(days=d)
    return f"{day.month:02d}/{day.day:02d}({DOW[d % 7]}) {r // 60:02d}:{r % 60:02d}"

def check_rest(df):
    """근무가 끝난 뒤 다음 일과·교육·근무가 시작될 때까지의 휴식 시간이 기준 이상인지 점검한다."""
    work = df[(df["kind"].isin(["duty", "training"])) | ((df["kind"] == "routine") & (df["sub"] == "work"))]
    rows = []
    for d in df[df["kind"] == "duty"].itertuples():
        need = REST_RULE.get(d.sub, 0) * 60
        nxt = work[work["start"] >= d.end]
        if nxt.empty:
            continue
        n = nxt.iloc[0]
        rest = int(n["start"] - d.end)
        rows.append({"근무": f"{d.id} {d.name}", "종료": fmt(d.end), "다음 일정": n["name"], "다음 시작": fmt(n["start"]),
                     "휴식(시간)": round(rest / 60, 1), "기준(시간)": need / 60,
                     "판정": "적정" if rest >= need else "위반", "부족(분)": max(0, need - rest), "id": d.id})
    return pd.DataFrame(rows)

def day_window(d):
    return (t(d, "06:30") if d % 7 < 5 else t(d, "07:30")), t(d, DAY_END)


def merge(iv):
    out = []
    for s, e in sorted(iv):
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out


def free_intervals(df, min_len=30, protect=True):
    """일과·근무·교육·휴가를 모두 뺀 빈 시간 구간. protect=True면 근무 후 휴식 구간도 뺀다."""
    busy = [(r.start, r.end) for r in df.itertuples()]
    if protect:
        busy += [(r.end, r.end + REST_RULE.get(r.sub, 0) * 60) for r in df[df["kind"] == "duty"].itertuples()]
    busy = merge(busy)
    free = []
    for d in range(14):
        lo, hi = day_window(d)
        cur = lo
        for s, e in busy:
            if e <= cur or s >= hi:
                continue
            if s > cur:
                free.append((cur, min(s, hi)))
            cur = max(cur, e)
        if cur < hi:
            free.append((cur, hi))
    return [(s, e) for s, e in free if e - s >= min_len]

SLOT = 30                                   # 계획은 30분 칸 단위
PW = {1: 1.0, 2: 2.0, 3: 3.0, 4: 4.0, 5: 5.0}   # 우선순위 가중치


def cand_blocks(free, g):
    """목표 g가 들어갈 수 있는 모든 후보 블록(시작, 종료)."""
    dl = t(*g["deadline"])
    out = []
    for s, e in free:
        d = s // 1440
        a, b = max(s, d * 1440 + g["allow"][0]), min(e, d * 1440 + g["allow"][1], dl)
        for st in range(-(-a // SLOT) * SLOT, b, SLOT):
            for L in g["lens"]:
                if st + L <= b:
                    out.append((g["id"], st, st + L))
    return out


def block_value(g, st, en, mode="priority"):
    L = en - st
    v = (PW[g["priority"]] if mode == "priority" else 1.0) * L
    if g["pref"] and g["pref"][0] <= st % 1440 and (en - 1) % 1440 < g["pref"][1]:
        v += 0.2 * L                        # 선호 시간대(저녁) 가산
    return v - 5 - 0.0005 * st              # 블록 수는 적게, 같은 값이면 이른 시각 우선


def plan_greedy(free, goals, day_total_cap=240):
    """탐욕: 우선순위 높은 목표부터, 이른 빈 시간부터 차례로 채운다."""
    ok = set()
    for s, e in free:
        ok.update(range(s // SLOT, e // SLOT))
    order = sorted(goals, key=lambda g: (-g["priority"], t(*g["deadline"])))
    used, blocks = {}, []
    for g in order:
        remain, dl = g["target"], t(*g["deadline"]) // SLOT
        lo, hi, mn, mx = g["allow"][0] // SLOT, g["allow"][1] // SLOT, min(g["lens"]), max(g["lens"])
        for d in range(14):
            cap = min(g["day_cap"], day_total_cap - used.get(d, 0), remain)
            k, end = d * 48 + lo, min(d * 48 + hi, dl)
            while remain > 0 and cap >= mn and k < end:
                if k not in ok:
                    k += 1
                    continue
                run = 0
                while k + run < end and (k + run) in ok:
                    run += 1
                L = min(mx, run * SLOT, cap) // SLOT * SLOT
                if L >= mn:
                    blocks.append((g["id"], k * SLOT, k * SLOT + L))
                    ok.difference_update(range(k, k + L // SLOT))
                    cap -= L; remain -= L; used[d] = used.get(d, 0) + L
                    k += L // SLOT
                else:
                    k += run
    return sorted(blocks, key=lambda b: b[1])


def plan_milp(free, goals, mode="priority", day_total_cap=240, prev=None, bonus=0.0):
    """최적화: 후보 블록 중 겹치지 않게 고르되 목표 시간·하루 상한·마감을 지키며 가치 합을 최대로 한다."""
    G = {g["id"]: g for g in goals}
    cands = [c for g in goals for c in cand_blocks(free, g)]
    if not cands:
        return []
    groups = {}
    for i, (gid, s, e) in enumerate(cands):
        for k in range(s // SLOT, e // SLOT):
            groups.setdefault(("slot", k), (1, []))[1].append((i, 1))
        groups.setdefault(("goal", gid), (G[gid]["target"], []))[1].append((i, e - s))
        groups.setdefault(("gday", gid, s // 1440), (G[gid]["day_cap"], []))[1].append((i, e - s))
        groups.setdefault(("day", s // 1440), (day_total_cap, []))[1].append((i, e - s))
    r, c, v, ub = [], [], [], []
    for row, (cap, items) in enumerate(groups.values()):
        ub.append(cap)
        for i, coef in items:
            r.append(row); c.append(i); v.append(coef)
    A = coo_matrix((v, (r, c)), shape=(len(ub), len(cands)))
    val = np.array([block_value(G[g], s, e, mode) + (bonus * (e - s) if prev and (g, s, e) in prev else 0)
                    for g, s, e in cands])
    res = milp(-val, constraints=LinearConstraint(A, -np.inf, np.array(ub, float)),
               integrality=np.ones(len(cands)), bounds=Bounds(0, 1), options={"time_limit": 20})
    if res.x is None:
        return []
    return sorted([cands[i] for i in range(len(cands)) if res.x[i] > 0.5], key=lambda b: b[1])


def evaluate(blocks, goals):
    rows = []
    for g in goals:
        bs = [b for b in blocks if b[0] == g["id"]]
        got = sum(e - s for _, s, e in bs)
        late = sum(1 for _, s, e in bs if e > t(*g["deadline"]))
        rows.append({"목표": g["name"], "요구(분)": g["target"], "배정(분)": got, "달성률": f"{got / g['target']:.0%}",
                     "블록 수": len(bs), "마감 초과": late,
                     "판정": "목표 달성" if got >= g["target"] and late == 0 else "미달"})
    return pd.DataFrame(rows)


def blocks_outside_free(blocks, free):
    """빈 시간 밖(다른 일정이나 휴식 보호 구간과 겹침)에 놓인 블록 수"""
    return sum(1 for _, s, e in blocks if not any(fs <= s and e <= fe for fs, fe in free))

import copy

df = to_df(EVENTS)
free = free_intervals(df)

# 실험 1. 하루 개인 시간 상한을 120분으로 줄여 시간이 모자라게 만든다 (요구 1260분 > 가능 약 1200분)
for title, goals, mode in (("우선순위 무시(모든 목표 가중치 같음)", GOALS, "equal"), ("우선순위 반영", GOALS, "priority")):
    bl = plan_milp(free, goals, mode=mode, day_total_cap=120)
    ev = evaluate(bl, goals)
    print(f"[{title}] 배정 {sum(e - s for _, s, e in bl)}분")
    print(ev[["목표", "요구(분)", "배정(분)", "달성률", "판정"]].to_string(index=False), "\n")

# 실험 2. 영어 학습의 우선순위를 5로 올리면 자격증 공부와 자리를 어떻게 나누는가
g2 = copy.deepcopy(GOALS)
g2[2]["priority"] = 5
bl = plan_milp(free, g2, day_total_cap=120)
print("[영어 우선순위 5로 변경]")
print(evaluate(bl, g2)[["목표", "배정(분)", "달성률"]].to_string(index=False), "\n")

# 실험 3. 자격증 공부 마감을 금요일(11/06) 24:00로 당기면 몇 분까지 가능한가
g3 = copy.deepcopy(GOALS)
g3[0]["deadline"] = (4, "24:00")
alone = plan_milp(free, [g3[0]])
got = sum(e - s for _, s, e in alone)
late = sum(1 for _, s, e in alone if e > t(*g3[0]["deadline"]))
print(f"[마감 11/06(금) 24:00] 자격증 공부 최대 {got}분 배정 (요구 {g3[0]['target']}분), 마감 초과 블록 {late}개")
print("부족", g3[0]["target"] - got, "분 -> 마감 연장 또는 목표 시간 축소 필요 (결정은 본인과 분대장)")

# 마감 점검 함수 시험: 마감 뒤 블록을 일부러 넣어 본다
fake = alone + [("G1", t(6, "10:00"), t(6, "11:00"))]
print("마감 초과 탐지 시험:", evaluate(fake, g3).loc[0, "마감 초과"], "건")


---

## 🎯 미션 07. 일정 변경 공지 반영하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **일정 변경 공지(이동·연장·취소)를 반영해 충돌·휴식·빈 시간을 다시 계산합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **변경 공지 4건과 대상이 없는 시험용 공지 1건을 일정에 반영하고 반영 기록을 남깁니다 과정을 진행하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `변경 반영 기록, 전후 비교, 재배정 결과` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [선택] 변경 반영·재계산 — 변경 공지를 반영해 전후를 비교하고 기존 계획을 유지하며 다시 배정했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(변경 반영 기록, 전후 비교, 재배정 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""일정 변경 공지(이동·연장·취소)를 반영해 충돌·휴식·빈 시간을 다시 계산하고, 기존 개인 계획은 최대한 유지하며 다시 배정한다."""
import re
import datetime as dt
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import coo_matrix

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 2주(11/02~11/15) 일정 자료 (실제 부대·인원·규정과 무관)
# 시각은 "일차(0=11/02 월요일) + HH:MM"으로 표기하며, 24시를 넘기면 26:00처럼 씁니다.
# ---------------------------------------------------------------------------
START = dt.date(2026, 11, 2)
DOW = "월화수목금토일"
DAY_END = "21:00"          # 개인 시간 계획은 저녁 점호(21:00) 전까지만 다룬다


def t(day, hhmm):
    # [TODO : 학생 실습 7] 일정 변경 공지 반영하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def E(eid, name, kind, day, s, e, end_day=None, sub=""):
    # [TODO : 학생 실습 7] 일정 변경 공지 반영하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_routine():
    ev = []
    for d in range(14):
        wd = d % 7
        if wd < 5:
            items = [("기상·점호", "fixed", "06:30", "07:00"), ("조식", "fixed", "07:00", "08:00"),
                     ("청소·정비", "fixed", "08:00", "09:00"), ("오전 일과", "work", "09:00", "12:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("오후 일과", "work", "13:00", "17:30"),
                     ("석식", "fixed", "17:30", "18:30"), ("저녁 점호", "fixed", "21:00", "21:30")]
        else:
            items = [("기상·점호", "fixed", "07:30", "08:00"), ("조식", "fixed", "08:00", "09:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("석식", "fixed", "17:30", "18:30"),
                     ("저녁 점호", "fixed", "21:00", "21:30")]
            if wd == 5:
                items.insert(2, ("주말 청소·정비", "work", "09:00", "10:30"))
        for k, (nm, sub, s, e) in enumerate(items):
            ev.append(E(f"R{d:02d}{k}", nm, "routine", d, s, e, sub=sub))
    return ev


EVENTS = make_routine() + [
    # 개인 근무표 (가상): 불침번·경계근무·당직 보조
    E("D01", "불침번", "duty", 0, "26:00", "28:00", sub="불침번"),
    E("D02", "불침번", "duty", 2, "22:00", "24:00", sub="불침번"),
    E("D03", "경계근무", "duty", 3, "16:00", "18:00", sub="경계근무"),
    E("D04", "불침번", "duty", 4, "26:00", "28:00", sub="불침번"),
    E("D05", "당직 보조", "duty", 6, "08:00", "32:00", sub="당직"),
    E("D06", "불침번", "duty", 8, "22:00", "24:00", sub="불침번"),
    E("D07", "불침번", "duty", 9, "22:00", "24:00", sub="불침번"),
    # 교육·훈련 일정
    E("T01", "정훈교육", "training", 1, "14:00", "16:00"),
    E("T02", "사이버 보안 교육", "training", 2, "20:30", "21:30"),
    E("T03", "정신교육", "training", 3, "15:00", "17:00"),
    E("T04", "사격 훈련", "training", 4, "09:00", "12:00"),
    E("T05", "동계 피복 보급", "training", 4, "13:00", "17:00"),
    E("T06", "보안 점검 교육", "training", 7, "15:00", "17:00"),
    E("T07", "체력검정", "training", 8, "10:00", "12:00"),
    E("T08", "신체검사(의무실)", "training", 8, "10:30", "11:30"),
    E("T09", "독감 예방접종", "training", 9, "09:00", "10:30"),
    E("T10", "분대 전술훈련(오전)", "training", 10, "09:00", "12:00"),
    E("T11", "분대 전술훈련(오후)", "training", 10, "13:00", "17:30"),
    # 휴가 (금요일 오전부터 일요일 저녁까지)
    E("V01", "정기 휴가", "leave", 11, "09:00", "18:00", end_day=13),
]

# 근무 종류별 근무 후 최소 휴식 시간(시간) - 교육용 가상 기준
REST_RULE = {"불침번": 6, "당직": 12, "경계근무": 2}

# 개인 목표: priority 1(낮음)~5(높음), allow=하루 중 허용 시간대(분), pref=선호 시간대(분)
GOALS = [
    {"id": "G1", "name": "자격증 공부", "target": 720, "lens": [60, 90, 120], "day_cap": 180,
     "deadline": (10, "24:00"), "priority": 3, "allow": (540, 1260), "pref": (1110, 1260)},
    {"id": "G2", "name": "체력단련", "target": 300, "lens": [60, 90], "day_cap": 90,
     "deadline": (13, "24:00"), "priority": 2, "allow": (360, 1260), "pref": None},
    {"id": "G3", "name": "영어 단어 학습", "target": 240, "lens": [30, 60], "day_cap": 60,
     "deadline": (13, "24:00"), "priority": 1, "allow": (540, 1260), "pref": None},
]

# 일정 변경 공지 (가상): 원래 일정을 옮기거나 늘리거나 취소한다
CHANGES = [
    {"id": "S01", "from": "교육훈련과", "title": "사격 훈련 연기(11/06 -> 11/13)", "op": "move", "target": "T04",
     "day": 11, "start": "09:00", "end": "12:00"},
    {"id": "S02", "from": "교육훈련과", "title": "체력검정 일정 변경(11/10 -> 11/11)", "op": "move", "target": "T07",
     "day": 9, "start": "10:00", "end": "12:00"},
    {"id": "S03", "from": "교육훈련과", "title": "분대 전술훈련 오후 일정 야간 연장", "op": "extend", "target": "T11", "end": "21:00"},
    {"id": "S04", "from": "작전과", "title": "경계근무 취소", "op": "cancel", "target": "D03"},
]

KINDS = {"routine": "부대 일과", "duty": "근무", "training": "교육·훈련", "leave": "휴가"}


def validate(ev):
    """일정 한 건의 형식 오류 목록을 돌려준다(빈 목록이면 정상)."""
    errs = [f"{k} 누락" for k in ("id", "name", "kind", "day", "start", "end") if ev.get(k) in (None, "")]
    if ev.get("kind") not in KINDS:
        errs.append("종류 오류")
    for k in ("start", "end"):
        if ev.get(k) and not re.match(r"^\d{1,2}:[0-5]\d$", str(ev[k])):
            errs.append(f"{k} 시간 형식 오류")
    if not errs:
        if not 0 <= ev["day"] < 14:
            errs.append("기간(0~13일) 밖")
        elif t(ev.get("end_day", ev["day"]), ev["end"]) <= t(ev["day"], ev["start"]):
            errs.append("종료가 시작보다 빠르거나 같음")
    return errs


def to_df(raw):
    """일정 목록을 '시작·종료 = 11/02 00:00부터의 분' 표로 바꾼다."""
    rows = []
    for ev in raw:
        s, e = t(ev["day"], ev["start"]), t(ev.get("end_day", ev["day"]), ev["end"])
        rows.append({"id": ev["id"], "name": ev["name"], "kind": ev["kind"], "sub": ev.get("sub", ""),
                     "start": s, "end": e, "dur": e - s})
    return pd.DataFrame(rows).sort_values(["start", "end"]).reset_index(drop=True)


def fmt(x):
    d, r = divmod(int(x), 1440)
    day = START + dt.timedelta(days=d)
    return f"{day.month:02d}/{day.day:02d}({DOW[d % 7]}) {r // 60:02d}:{r % 60:02d}"

def check_rest(df):
    """근무가 끝난 뒤 다음 일과·교육·근무가 시작될 때까지의 휴식 시간이 기준 이상인지 점검한다."""
    work = df[(df["kind"].isin(["duty", "training"])) | ((df["kind"] == "routine") & (df["sub"] == "work"))]
    rows = []
    for d in df[df["kind"] == "duty"].itertuples():
        need = REST_RULE.get(d.sub, 0) * 60
        nxt = work[work["start"] >= d.end]
        if nxt.empty:
            continue
        n = nxt.iloc[0]
        rest = int(n["start"] - d.end)
        rows.append({"근무": f"{d.id} {d.name}", "종료": fmt(d.end), "다음 일정": n["name"], "다음 시작": fmt(n["start"]),
                     "휴식(시간)": round(rest / 60, 1), "기준(시간)": need / 60,
                     "판정": "적정" if rest >= need else "위반", "부족(분)": max(0, need - rest), "id": d.id})
    return pd.DataFrame(rows)

def day_window(d):
    return (t(d, "06:30") if d % 7 < 5 else t(d, "07:30")), t(d, DAY_END)


def merge(iv):
    out = []
    for s, e in sorted(iv):
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out


def free_intervals(df, min_len=30, protect=True):
    """일과·근무·교육·휴가를 모두 뺀 빈 시간 구간. protect=True면 근무 후 휴식 구간도 뺀다."""
    busy = [(r.start, r.end) for r in df.itertuples()]
    if protect:
        busy += [(r.end, r.end + REST_RULE.get(r.sub, 0) * 60) for r in df[df["kind"] == "duty"].itertuples()]
    busy = merge(busy)
    free = []
    for d in range(14):
        lo, hi = day_window(d)
        cur = lo
        for s, e in busy:
            if e <= cur or s >= hi:
                continue
            if s > cur:
                free.append((cur, min(s, hi)))
            cur = max(cur, e)
        if cur < hi:
            free.append((cur, hi))
    return [(s, e) for s, e in free if e - s >= min_len]

SLOT = 30                                   # 계획은 30분 칸 단위
PW = {1: 1.0, 2: 2.0, 3: 3.0, 4: 4.0, 5: 5.0}   # 우선순위 가중치


def cand_blocks(free, g):
    """목표 g가 들어갈 수 있는 모든 후보 블록(시작, 종료)."""
    dl = t(*g["deadline"])
    out = []
    for s, e in free:
        d = s // 1440
        a, b = max(s, d * 1440 + g["allow"][0]), min(e, d * 1440 + g["allow"][1], dl)
        for st in range(-(-a // SLOT) * SLOT, b, SLOT):
            for L in g["lens"]:
                if st + L <= b:
                    out.append((g["id"], st, st + L))
    return out


def block_value(g, st, en, mode="priority"):
    L = en - st
    v = (PW[g["priority"]] if mode == "priority" else 1.0) * L
    if g["pref"] and g["pref"][0] <= st % 1440 and (en - 1) % 1440 < g["pref"][1]:
        v += 0.2 * L                        # 선호 시간대(저녁) 가산
    return v - 5 - 0.0005 * st              # 블록 수는 적게, 같은 값이면 이른 시각 우선


def plan_greedy(free, goals, day_total_cap=240):
    """탐욕: 우선순위 높은 목표부터, 이른 빈 시간부터 차례로 채운다."""
    ok = set()
    for s, e in free:
        ok.update(range(s // SLOT, e // SLOT))
    order = sorted(goals, key=lambda g: (-g["priority"], t(*g["deadline"])))
    used, blocks = {}, []
    for g in order:
        remain, dl = g["target"], t(*g["deadline"]) // SLOT
        lo, hi, mn, mx = g["allow"][0] // SLOT, g["allow"][1] // SLOT, min(g["lens"]), max(g["lens"])
        for d in range(14):
            cap = min(g["day_cap"], day_total_cap - used.get(d, 0), remain)
            k, end = d * 48 + lo, min(d * 48 + hi, dl)
            while remain > 0 and cap >= mn and k < end:
                if k not in ok:
                    k += 1
                    continue
                run = 0
                while k + run < end and (k + run) in ok:
                    run += 1
                L = min(mx, run * SLOT, cap) // SLOT * SLOT
                if L >= mn:
                    blocks.append((g["id"], k * SLOT, k * SLOT + L))
                    ok.difference_update(range(k, k + L // SLOT))
                    cap -= L; remain -= L; used[d] = used.get(d, 0) + L
                    k += L // SLOT
                else:
                    k += run
    return sorted(blocks, key=lambda b: b[1])


def plan_milp(free, goals, mode="priority", day_total_cap=240, prev=None, bonus=0.0):
    """최적화: 후보 블록 중 겹치지 않게 고르되 목표 시간·하루 상한·마감을 지키며 가치 합을 최대로 한다."""
    G = {g["id"]: g for g in goals}
    cands = [c for g in goals for c in cand_blocks(free, g)]
    if not cands:
        return []
    groups = {}
    for i, (gid, s, e) in enumerate(cands):
        for k in range(s // SLOT, e // SLOT):
            groups.setdefault(("slot", k), (1, []))[1].append((i, 1))
        groups.setdefault(("goal", gid), (G[gid]["target"], []))[1].append((i, e - s))
        groups.setdefault(("gday", gid, s // 1440), (G[gid]["day_cap"], []))[1].append((i, e - s))
        groups.setdefault(("day", s // 1440), (day_total_cap, []))[1].append((i, e - s))
    r, c, v, ub = [], [], [], []
    for row, (cap, items) in enumerate(groups.values()):
        ub.append(cap)
        for i, coef in items:
            r.append(row); c.append(i); v.append(coef)
    A = coo_matrix((v, (r, c)), shape=(len(ub), len(cands)))
    val = np.array([block_value(G[g], s, e, mode) + (bonus * (e - s) if prev and (g, s, e) in prev else 0)
                    for g, s, e in cands])
    res = milp(-val, constraints=LinearConstraint(A, -np.inf, np.array(ub, float)),
               integrality=np.ones(len(cands)), bounds=Bounds(0, 1), options={"time_limit": 20})
    if res.x is None:
        return []
    return sorted([cands[i] for i in range(len(cands)) if res.x[i] > 0.5], key=lambda b: b[1])


def evaluate(blocks, goals):
    rows = []
    for g in goals:
        bs = [b for b in blocks if b[0] == g["id"]]
        got = sum(e - s for _, s, e in bs)
        late = sum(1 for _, s, e in bs if e > t(*g["deadline"]))
        rows.append({"목표": g["name"], "요구(분)": g["target"], "배정(분)": got, "달성률": f"{got / g['target']:.0%}",
                     "블록 수": len(bs), "마감 초과": late,
                     "판정": "목표 달성" if got >= g["target"] and late == 0 else "미달"})
    return pd.DataFrame(rows)


def blocks_outside_free(blocks, free):
    """빈 시간 밖(다른 일정이나 휴식 보호 구간과 겹침)에 놓인 블록 수"""
    return sum(1 for _, s, e in blocks if not any(fs <= s and e <= fe for fs, fe in free))

def classify(a, b):
    """겹친 두 일정의 성격: hard(실제 충돌) / soft(식사·점호 침범) / replace(일과 대체) / exempt(근무·휴가자 일과 열외)"""
    if a["kind"] != "routine" and b["kind"] != "routine":
        return "hard"
    r, o = (a, b) if a["kind"] == "routine" else (b, a)
    if o["kind"] == "routine":
        return "hard"
    if o["kind"] in ("duty", "leave"):
        return "exempt"
    return "replace" if r["sub"] == "work" else "soft"


def find_conflicts(df):
    """시작 시각순으로 정렬된 일정에서 겹치는 쌍을 찾는다(끝난 일정 뒤로는 비교하지 않음)."""
    rows = df.to_dict("records")
    out = []
    for i, a in enumerate(rows):
        for b in rows[i + 1:]:
            if b["start"] >= a["end"]:
                break
            out.append({"일정A": f"{a['id']} {a['name']}", "일정B": f"{b['id']} {b['name']}",
                        "겹침(분)": min(a["end"], b["end"]) - b["start"], "시작": b["start"],
                        "구분": classify(a, b)})
    return pd.DataFrame(out, columns=["일정A", "일정B", "겹침(분)", "시작", "구분"])

def day_label(d):
    x = START + dt.timedelta(days=d)
    return f"{x.month:02d}/{x.day:02d}({DOW[d % 7]})"


def rel(x, lo):
    v = int(x - lo)
    return f"{v // 60:02d}:{v % 60:02d}"


def timetable(df, blocks, goals, days, free):
    """하루 단위 일정표: 근무·교육·휴가와 개인 목표 배정, 남는 빈 시간."""
    names = {g["id"]: g["name"] for g in goals}
    rows = []
    for d in days:
        lo, hi = d * 1440, (d + 1) * 1440
        part = df[(df["end"] > lo) & (df["start"] < hi) & (df["kind"] != "routine")]
        fixed = [f"{r.name} {rel(max(r.start, lo), lo)}~{rel(min(r.end, hi), lo)}" for r in part.itertuples()]
        mine = [b for b in blocks if lo <= b[1] < hi]
        pers = [f"{names[g]} {rel(s, lo)}~{rel(e, lo)}" for g, s, e in mine]
        free_min = sum(min(e, hi) - max(s, lo) for s, e in free if e > lo and s < hi) - sum(e - s for _, s, e in mine)
        rows.append({"날짜": day_label(d), "근무·교육·휴가": ", ".join(fixed) or "-",
                     "개인 목표 배정": ", ".join(pers) or "-", "남는 빈 시간(분)": free_min})
    return pd.DataFrame(rows)


def notify(df, blocks, goals, rest, d, conf=None):
    """다음 날(d일차) 알림 문구. 모든 문구는 AI의 제안이며 확정은 사람이 한다."""
    names = {g["id"]: g["name"] for g in goals}
    lo, hi = d * 1440, (d + 1) * 1440
    lines = [f"[일정 알림 - AI 제안] {day_label(d)} 일정"]
    for r in df[(df["start"] >= lo) & (df["start"] < hi) & (df["kind"] == "duty")].itertuples():
        w = rest[rest["id"] == r.id] if len(rest) else rest
        tail = ""
        if len(w) and w.iloc[0]["판정"] == "위반":
            tail = f" (근무 후 휴식 {w.iloc[0]['휴식(시간)']}시간으로 기준 {w.iloc[0]['기준(시간)']:.0f}시간보다 짧음 - 분대장 확인 필요)"
        lines.append(f"- 근무: {r.name} {rel(r.start, lo)}~{rel(r.end, lo) if r.end <= hi else '익일 ' + rel(r.end, hi)}{tail}")
    for r in df[(df["end"] > lo) & (df["start"] < hi) & (df["kind"].isin(["training", "leave"]))].itertuples():
        lines.append(f"- {KINDS[r.kind]}: {r.name} {rel(max(r.start, lo), lo)}~{rel(min(r.end, hi), lo)}")
    mine = [b for b in blocks if lo <= b[1] < hi]
    for g, s, e in mine:
        lines.append(f"- 개인 목표 제안: {rel(s, lo)}~{rel(e, lo)} {names[g]} ({e - s}분)")
    if conf is not None and len(conf):
        for c in conf[(conf["구분"] == "hard") & (conf["시작"] >= lo) & (conf["시작"] < hi)].to_dict("records"):
            lines.append(f"- 확인 필요: {c['일정A']} / {c['일정B']} {c['겹침(분)']}분 겹침")
    if len(lines) == 1:
        lines.append("- 특이 일정 없음")
    lines.append("※ 근무 편성과 일정 확정은 분대장·행정반이 합니다. 이 알림은 제안입니다.")
    return "\n".join(lines)


def apply_changes(raw, changes):
    """변경 공지를 일정 목록에 반영한다. 대상 일정이 없거나 형식이 잘못된 공지는 건너뛴다."""
    ev, log = {e["id"]: dict(e) for e in raw}, []
    for c in changes:
        e = ev.get(c.get("target"))
        if e is None:
            log.append(f"{c['id']}: 대상 일정 없음 - 반영 안 함")
        elif c["op"] == "cancel":
            del ev[c["target"]]; log.append(f"{c['id']}: {e['name']} 취소")
        elif c["op"] == "move":
            e.update(day=c["day"], end_day=c["day"], start=c["start"], end=c["end"]); log.append(f"{c['id']}: {e['name']} 이동 -> {day_label(c['day'])} {c['start']}~{c['end']}")
        elif c["op"] == "extend":
            e["end"] = c["end"]; log.append(f"{c['id']}: {e['name']} 종료 {c['end']}로 연장")
        else:
            log.append(f"{c['id']}: 알 수 없는 변경 유형 - 반영 안 함")
    return list(ev.values()), log

df0 = to_df(EVENTS)
free0 = free_intervals(df0)
plan0 = plan_milp(free0, GOALS)

bad = CHANGES + [{"id": "S99", "from": "시험", "title": "없는 일정 이동", "op": "move", "target": "T99", "day": 1, "start": "09:00", "end": "10:00"}]
events1, log = apply_changes(EVENTS, bad)
print("[변경 공지 반영 기록]")
for x in log:
    print("-", x)

df1 = to_df(events1)
c0, c1 = find_conflicts(df0), find_conflicts(df1)
h0, h1 = set(c0[c0["구분"] == "hard"][["일정A", "일정B"]].apply(tuple, axis=1)), set(c1[c1["구분"] == "hard"][["일정A", "일정B"]].apply(tuple, axis=1))
print(f"\n실제 충돌: 변경 전 {len(h0)}건 -> 변경 후 {len(h1)}건")
print(" 해소:", sorted(h0 - h1))
print(" 새로 생김:", sorted(h1 - h0))
print("식사·점호 침범:", (c0["구분"] == "soft").sum(), "->", (c1["구분"] == "soft").sum())
print("휴식 위반:", (check_rest(df0)["판정"] == "위반").sum(), "->", (check_rest(df1)["판정"] == "위반").sum())

free1 = free_intervals(df1)
broken = [b for b in plan0 if blocks_outside_free([b], free1)]
print(f"\n빈 시간 {sum(e - s for s, e in free0)}분 -> {sum(e - s for s, e in free1)}분, 기존 계획 중 무효가 된 블록 {len(broken)}개")

# 처음부터 다시 vs 기존 계획 유지 가산점을 주고 다시
fresh = plan_milp(free1, GOALS)
keep = plan_milp(free1, GOALS, prev=set(plan0), bonus=1.0)
for title, bl in (("처음부터 다시 계산", fresh), ("기존 계획 유지 가산", keep)):
    kept = len(set(bl) & set(plan0))
    print(f"[{title}] 유지 {kept}/{len(plan0)}블록, 달성 {list(evaluate(bl, GOALS)['판정']).count('목표 달성')}/3, 충돌 {blocks_outside_free(bl, free1)}개")
print("\n목표 달성" if blocks_outside_free(keep, free1) == 0 and len(set(keep) & set(plan0)) > len(set(fresh) & set(plan0)) - 1 else "\n미달")


---

## 🎯 미션 08. 주간 일정표·알림 만들기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **개인 계획과 고정 일정을 합쳐 주간 일정표, 간트 그래프, 알림 문구를 만듭니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **날짜별로 근무·교육·휴가와 개인 목표 배정, 남는 빈 시간을 표로 정리하고 체계화하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `주간 일정표, 간트 그래프, 알림 문구` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 일정표·알림 생성 — 읽기 쉬운 일정표와 그래프, 확정 주체를 밝힌 알림 문구를 자동 생성했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(주간 일정표, 간트 그래프, 알림 문구)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""최적화된 개인 계획과 고정 일정을 합쳐 주간 일정표(표·간트 그래프)와 다음 날 알림 문구를 자동으로 만든다."""
import re
import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import coo_matrix

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 2주(11/02~11/15) 일정 자료 (실제 부대·인원·규정과 무관)
# 시각은 "일차(0=11/02 월요일) + HH:MM"으로 표기하며, 24시를 넘기면 26:00처럼 씁니다.
# ---------------------------------------------------------------------------
START = dt.date(2026, 11, 2)
DOW = "월화수목금토일"
DAY_END = "21:00"          # 개인 시간 계획은 저녁 점호(21:00) 전까지만 다룬다


def t(day, hhmm):
    # [TODO : 학생 실습 8] 주간 일정표·알림 만들기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def E(eid, name, kind, day, s, e, end_day=None, sub=""):
    # [TODO : 학생 실습 8] 주간 일정표·알림 만들기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_routine():
    ev = []
    for d in range(14):
        wd = d % 7
        if wd < 5:
            items = [("기상·점호", "fixed", "06:30", "07:00"), ("조식", "fixed", "07:00", "08:00"),
                     ("청소·정비", "fixed", "08:00", "09:00"), ("오전 일과", "work", "09:00", "12:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("오후 일과", "work", "13:00", "17:30"),
                     ("석식", "fixed", "17:30", "18:30"), ("저녁 점호", "fixed", "21:00", "21:30")]
        else:
            items = [("기상·점호", "fixed", "07:30", "08:00"), ("조식", "fixed", "08:00", "09:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("석식", "fixed", "17:30", "18:30"),
                     ("저녁 점호", "fixed", "21:00", "21:30")]
            if wd == 5:
                items.insert(2, ("주말 청소·정비", "work", "09:00", "10:30"))
        for k, (nm, sub, s, e) in enumerate(items):
            ev.append(E(f"R{d:02d}{k}", nm, "routine", d, s, e, sub=sub))
    return ev


EVENTS = make_routine() + [
    # 개인 근무표 (가상): 불침번·경계근무·당직 보조
    E("D01", "불침번", "duty", 0, "26:00", "28:00", sub="불침번"),
    E("D02", "불침번", "duty", 2, "22:00", "24:00", sub="불침번"),
    E("D03", "경계근무", "duty", 3, "16:00", "18:00", sub="경계근무"),
    E("D04", "불침번", "duty", 4, "26:00", "28:00", sub="불침번"),
    E("D05", "당직 보조", "duty", 6, "08:00", "32:00", sub="당직"),
    E("D06", "불침번", "duty", 8, "22:00", "24:00", sub="불침번"),
    E("D07", "불침번", "duty", 9, "22:00", "24:00", sub="불침번"),
    # 교육·훈련 일정
    E("T01", "정훈교육", "training", 1, "14:00", "16:00"),
    E("T02", "사이버 보안 교육", "training", 2, "20:30", "21:30"),
    E("T03", "정신교육", "training", 3, "15:00", "17:00"),
    E("T04", "사격 훈련", "training", 4, "09:00", "12:00"),
    E("T05", "동계 피복 보급", "training", 4, "13:00", "17:00"),
    E("T06", "보안 점검 교육", "training", 7, "15:00", "17:00"),
    E("T07", "체력검정", "training", 8, "10:00", "12:00"),
    E("T08", "신체검사(의무실)", "training", 8, "10:30", "11:30"),
    E("T09", "독감 예방접종", "training", 9, "09:00", "10:30"),
    E("T10", "분대 전술훈련(오전)", "training", 10, "09:00", "12:00"),
    E("T11", "분대 전술훈련(오후)", "training", 10, "13:00", "17:30"),
    # 휴가 (금요일 오전부터 일요일 저녁까지)
    E("V01", "정기 휴가", "leave", 11, "09:00", "18:00", end_day=13),
]

# 근무 종류별 근무 후 최소 휴식 시간(시간) - 교육용 가상 기준
REST_RULE = {"불침번": 6, "당직": 12, "경계근무": 2}

# 개인 목표: priority 1(낮음)~5(높음), allow=하루 중 허용 시간대(분), pref=선호 시간대(분)
GOALS = [
    {"id": "G1", "name": "자격증 공부", "target": 720, "lens": [60, 90, 120], "day_cap": 180,
     "deadline": (10, "24:00"), "priority": 3, "allow": (540, 1260), "pref": (1110, 1260)},
    {"id": "G2", "name": "체력단련", "target": 300, "lens": [60, 90], "day_cap": 90,
     "deadline": (13, "24:00"), "priority": 2, "allow": (360, 1260), "pref": None},
    {"id": "G3", "name": "영어 단어 학습", "target": 240, "lens": [30, 60], "day_cap": 60,
     "deadline": (13, "24:00"), "priority": 1, "allow": (540, 1260), "pref": None},
]

# 일정 변경 공지 (가상): 원래 일정을 옮기거나 늘리거나 취소한다
CHANGES = [
    {"id": "S01", "from": "교육훈련과", "title": "사격 훈련 연기(11/06 -> 11/13)", "op": "move", "target": "T04",
     "day": 11, "start": "09:00", "end": "12:00"},
    {"id": "S02", "from": "교육훈련과", "title": "체력검정 일정 변경(11/10 -> 11/11)", "op": "move", "target": "T07",
     "day": 9, "start": "10:00", "end": "12:00"},
    {"id": "S03", "from": "교육훈련과", "title": "분대 전술훈련 오후 일정 야간 연장", "op": "extend", "target": "T11", "end": "21:00"},
    {"id": "S04", "from": "작전과", "title": "경계근무 취소", "op": "cancel", "target": "D03"},
]

KINDS = {"routine": "부대 일과", "duty": "근무", "training": "교육·훈련", "leave": "휴가"}


def validate(ev):
    """일정 한 건의 형식 오류 목록을 돌려준다(빈 목록이면 정상)."""
    errs = [f"{k} 누락" for k in ("id", "name", "kind", "day", "start", "end") if ev.get(k) in (None, "")]
    if ev.get("kind") not in KINDS:
        errs.append("종류 오류")
    for k in ("start", "end"):
        if ev.get(k) and not re.match(r"^\d{1,2}:[0-5]\d$", str(ev[k])):
            errs.append(f"{k} 시간 형식 오류")
    if not errs:
        if not 0 <= ev["day"] < 14:
            errs.append("기간(0~13일) 밖")
        elif t(ev.get("end_day", ev["day"]), ev["end"]) <= t(ev["day"], ev["start"]):
            errs.append("종료가 시작보다 빠르거나 같음")
    return errs


def to_df(raw):
    """일정 목록을 '시작·종료 = 11/02 00:00부터의 분' 표로 바꾼다."""
    rows = []
    for ev in raw:
        s, e = t(ev["day"], ev["start"]), t(ev.get("end_day", ev["day"]), ev["end"])
        rows.append({"id": ev["id"], "name": ev["name"], "kind": ev["kind"], "sub": ev.get("sub", ""),
                     "start": s, "end": e, "dur": e - s})
    return pd.DataFrame(rows).sort_values(["start", "end"]).reset_index(drop=True)


def fmt(x):
    d, r = divmod(int(x), 1440)
    day = START + dt.timedelta(days=d)
    return f"{day.month:02d}/{day.day:02d}({DOW[d % 7]}) {r // 60:02d}:{r % 60:02d}"

def check_rest(df):
    """근무가 끝난 뒤 다음 일과·교육·근무가 시작될 때까지의 휴식 시간이 기준 이상인지 점검한다."""
    work = df[(df["kind"].isin(["duty", "training"])) | ((df["kind"] == "routine") & (df["sub"] == "work"))]
    rows = []
    for d in df[df["kind"] == "duty"].itertuples():
        need = REST_RULE.get(d.sub, 0) * 60
        nxt = work[work["start"] >= d.end]
        if nxt.empty:
            continue
        n = nxt.iloc[0]
        rest = int(n["start"] - d.end)
        rows.append({"근무": f"{d.id} {d.name}", "종료": fmt(d.end), "다음 일정": n["name"], "다음 시작": fmt(n["start"]),
                     "휴식(시간)": round(rest / 60, 1), "기준(시간)": need / 60,
                     "판정": "적정" if rest >= need else "위반", "부족(분)": max(0, need - rest), "id": d.id})
    return pd.DataFrame(rows)

def day_window(d):
    return (t(d, "06:30") if d % 7 < 5 else t(d, "07:30")), t(d, DAY_END)


def merge(iv):
    out = []
    for s, e in sorted(iv):
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out


def free_intervals(df, min_len=30, protect=True):
    """일과·근무·교육·휴가를 모두 뺀 빈 시간 구간. protect=True면 근무 후 휴식 구간도 뺀다."""
    busy = [(r.start, r.end) for r in df.itertuples()]
    if protect:
        busy += [(r.end, r.end + REST_RULE.get(r.sub, 0) * 60) for r in df[df["kind"] == "duty"].itertuples()]
    busy = merge(busy)
    free = []
    for d in range(14):
        lo, hi = day_window(d)
        cur = lo
        for s, e in busy:
            if e <= cur or s >= hi:
                continue
            if s > cur:
                free.append((cur, min(s, hi)))
            cur = max(cur, e)
        if cur < hi:
            free.append((cur, hi))
    return [(s, e) for s, e in free if e - s >= min_len]

SLOT = 30                                   # 계획은 30분 칸 단위
PW = {1: 1.0, 2: 2.0, 3: 3.0, 4: 4.0, 5: 5.0}   # 우선순위 가중치


def cand_blocks(free, g):
    """목표 g가 들어갈 수 있는 모든 후보 블록(시작, 종료)."""
    dl = t(*g["deadline"])
    out = []
    for s, e in free:
        d = s // 1440
        a, b = max(s, d * 1440 + g["allow"][0]), min(e, d * 1440 + g["allow"][1], dl)
        for st in range(-(-a // SLOT) * SLOT, b, SLOT):
            for L in g["lens"]:
                if st + L <= b:
                    out.append((g["id"], st, st + L))
    return out


def block_value(g, st, en, mode="priority"):
    L = en - st
    v = (PW[g["priority"]] if mode == "priority" else 1.0) * L
    if g["pref"] and g["pref"][0] <= st % 1440 and (en - 1) % 1440 < g["pref"][1]:
        v += 0.2 * L                        # 선호 시간대(저녁) 가산
    return v - 5 - 0.0005 * st              # 블록 수는 적게, 같은 값이면 이른 시각 우선


def plan_greedy(free, goals, day_total_cap=240):
    """탐욕: 우선순위 높은 목표부터, 이른 빈 시간부터 차례로 채운다."""
    ok = set()
    for s, e in free:
        ok.update(range(s // SLOT, e // SLOT))
    order = sorted(goals, key=lambda g: (-g["priority"], t(*g["deadline"])))
    used, blocks = {}, []
    for g in order:
        remain, dl = g["target"], t(*g["deadline"]) // SLOT
        lo, hi, mn, mx = g["allow"][0] // SLOT, g["allow"][1] // SLOT, min(g["lens"]), max(g["lens"])
        for d in range(14):
            cap = min(g["day_cap"], day_total_cap - used.get(d, 0), remain)
            k, end = d * 48 + lo, min(d * 48 + hi, dl)
            while remain > 0 and cap >= mn and k < end:
                if k not in ok:
                    k += 1
                    continue
                run = 0
                while k + run < end and (k + run) in ok:
                    run += 1
                L = min(mx, run * SLOT, cap) // SLOT * SLOT
                if L >= mn:
                    blocks.append((g["id"], k * SLOT, k * SLOT + L))
                    ok.difference_update(range(k, k + L // SLOT))
                    cap -= L; remain -= L; used[d] = used.get(d, 0) + L
                    k += L // SLOT
                else:
                    k += run
    return sorted(blocks, key=lambda b: b[1])


def plan_milp(free, goals, mode="priority", day_total_cap=240, prev=None, bonus=0.0):
    """최적화: 후보 블록 중 겹치지 않게 고르되 목표 시간·하루 상한·마감을 지키며 가치 합을 최대로 한다."""
    G = {g["id"]: g for g in goals}
    cands = [c for g in goals for c in cand_blocks(free, g)]
    if not cands:
        return []
    groups = {}
    for i, (gid, s, e) in enumerate(cands):
        for k in range(s // SLOT, e // SLOT):
            groups.setdefault(("slot", k), (1, []))[1].append((i, 1))
        groups.setdefault(("goal", gid), (G[gid]["target"], []))[1].append((i, e - s))
        groups.setdefault(("gday", gid, s // 1440), (G[gid]["day_cap"], []))[1].append((i, e - s))
        groups.setdefault(("day", s // 1440), (day_total_cap, []))[1].append((i, e - s))
    r, c, v, ub = [], [], [], []
    for row, (cap, items) in enumerate(groups.values()):
        ub.append(cap)
        for i, coef in items:
            r.append(row); c.append(i); v.append(coef)
    A = coo_matrix((v, (r, c)), shape=(len(ub), len(cands)))
    val = np.array([block_value(G[g], s, e, mode) + (bonus * (e - s) if prev and (g, s, e) in prev else 0)
                    for g, s, e in cands])
    res = milp(-val, constraints=LinearConstraint(A, -np.inf, np.array(ub, float)),
               integrality=np.ones(len(cands)), bounds=Bounds(0, 1), options={"time_limit": 20})
    if res.x is None:
        return []
    return sorted([cands[i] for i in range(len(cands)) if res.x[i] > 0.5], key=lambda b: b[1])


def evaluate(blocks, goals):
    rows = []
    for g in goals:
        bs = [b for b in blocks if b[0] == g["id"]]
        got = sum(e - s for _, s, e in bs)
        late = sum(1 for _, s, e in bs if e > t(*g["deadline"]))
        rows.append({"목표": g["name"], "요구(분)": g["target"], "배정(분)": got, "달성률": f"{got / g['target']:.0%}",
                     "블록 수": len(bs), "마감 초과": late,
                     "판정": "목표 달성" if got >= g["target"] and late == 0 else "미달"})
    return pd.DataFrame(rows)


def blocks_outside_free(blocks, free):
    """빈 시간 밖(다른 일정이나 휴식 보호 구간과 겹침)에 놓인 블록 수"""
    return sum(1 for _, s, e in blocks if not any(fs <= s and e <= fe for fs, fe in free))

def classify(a, b):
    """겹친 두 일정의 성격: hard(실제 충돌) / soft(식사·점호 침범) / replace(일과 대체) / exempt(근무·휴가자 일과 열외)"""
    if a["kind"] != "routine" and b["kind"] != "routine":
        return "hard"
    r, o = (a, b) if a["kind"] == "routine" else (b, a)
    if o["kind"] == "routine":
        return "hard"
    if o["kind"] in ("duty", "leave"):
        return "exempt"
    return "replace" if r["sub"] == "work" else "soft"


def find_conflicts(df):
    """시작 시각순으로 정렬된 일정에서 겹치는 쌍을 찾는다(끝난 일정 뒤로는 비교하지 않음)."""
    rows = df.to_dict("records")
    out = []
    for i, a in enumerate(rows):
        for b in rows[i + 1:]:
            if b["start"] >= a["end"]:
                break
            out.append({"일정A": f"{a['id']} {a['name']}", "일정B": f"{b['id']} {b['name']}",
                        "겹침(분)": min(a["end"], b["end"]) - b["start"], "시작": b["start"],
                        "구분": classify(a, b)})
    return pd.DataFrame(out, columns=["일정A", "일정B", "겹침(분)", "시작", "구분"])

def day_label(d):
    x = START + dt.timedelta(days=d)
    return f"{x.month:02d}/{x.day:02d}({DOW[d % 7]})"


def rel(x, lo):
    v = int(x - lo)
    return f"{v // 60:02d}:{v % 60:02d}"


def timetable(df, blocks, goals, days, free):
    """하루 단위 일정표: 근무·교육·휴가와 개인 목표 배정, 남는 빈 시간."""
    names = {g["id"]: g["name"] for g in goals}
    rows = []
    for d in days:
        lo, hi = d * 1440, (d + 1) * 1440
        part = df[(df["end"] > lo) & (df["start"] < hi) & (df["kind"] != "routine")]
        fixed = [f"{r.name} {rel(max(r.start, lo), lo)}~{rel(min(r.end, hi), lo)}" for r in part.itertuples()]
        mine = [b for b in blocks if lo <= b[1] < hi]
        pers = [f"{names[g]} {rel(s, lo)}~{rel(e, lo)}" for g, s, e in mine]
        free_min = sum(min(e, hi) - max(s, lo) for s, e in free if e > lo and s < hi) - sum(e - s for _, s, e in mine)
        rows.append({"날짜": day_label(d), "근무·교육·휴가": ", ".join(fixed) or "-",
                     "개인 목표 배정": ", ".join(pers) or "-", "남는 빈 시간(분)": free_min})
    return pd.DataFrame(rows)


def notify(df, blocks, goals, rest, d, conf=None):
    """다음 날(d일차) 알림 문구. 모든 문구는 AI의 제안이며 확정은 사람이 한다."""
    names = {g["id"]: g["name"] for g in goals}
    lo, hi = d * 1440, (d + 1) * 1440
    lines = [f"[일정 알림 - AI 제안] {day_label(d)} 일정"]
    for r in df[(df["start"] >= lo) & (df["start"] < hi) & (df["kind"] == "duty")].itertuples():
        w = rest[rest["id"] == r.id] if len(rest) else rest
        tail = ""
        if len(w) and w.iloc[0]["판정"] == "위반":
            tail = f" (근무 후 휴식 {w.iloc[0]['휴식(시간)']}시간으로 기준 {w.iloc[0]['기준(시간)']:.0f}시간보다 짧음 - 분대장 확인 필요)"
        lines.append(f"- 근무: {r.name} {rel(r.start, lo)}~{rel(r.end, lo) if r.end <= hi else '익일 ' + rel(r.end, hi)}{tail}")
    for r in df[(df["end"] > lo) & (df["start"] < hi) & (df["kind"].isin(["training", "leave"]))].itertuples():
        lines.append(f"- {KINDS[r.kind]}: {r.name} {rel(max(r.start, lo), lo)}~{rel(min(r.end, hi), lo)}")
    mine = [b for b in blocks if lo <= b[1] < hi]
    for g, s, e in mine:
        lines.append(f"- 개인 목표 제안: {rel(s, lo)}~{rel(e, lo)} {names[g]} ({e - s}분)")
    if conf is not None and len(conf):
        for c in conf[(conf["구분"] == "hard") & (conf["시작"] >= lo) & (conf["시작"] < hi)].to_dict("records"):
            lines.append(f"- 확인 필요: {c['일정A']} / {c['일정B']} {c['겹침(분)']}분 겹침")
    if len(lines) == 1:
        lines.append("- 특이 일정 없음")
    lines.append("※ 근무 편성과 일정 확정은 분대장·행정반이 합니다. 이 알림은 제안입니다.")
    return "\n".join(lines)


def apply_changes(raw, changes):
    """변경 공지를 일정 목록에 반영한다. 대상 일정이 없거나 형식이 잘못된 공지는 건너뛴다."""
    ev, log = {e["id"]: dict(e) for e in raw}, []
    for c in changes:
        e = ev.get(c.get("target"))
        if e is None:
            log.append(f"{c['id']}: 대상 일정 없음 - 반영 안 함")
        elif c["op"] == "cancel":
            del ev[c["target"]]; log.append(f"{c['id']}: {e['name']} 취소")
        elif c["op"] == "move":
            e.update(day=c["day"], end_day=c["day"], start=c["start"], end=c["end"]); log.append(f"{c['id']}: {e['name']} 이동 -> {day_label(c['day'])} {c['start']}~{c['end']}")
        elif c["op"] == "extend":
            e["end"] = c["end"]; log.append(f"{c['id']}: {e['name']} 종료 {c['end']}로 연장")
        else:
            log.append(f"{c['id']}: 알 수 없는 변경 유형 - 반영 안 함")
    return list(ev.values()), log

df = to_df(EVENTS)
free = free_intervals(df)
rest = check_rest(df)
conf = find_conflicts(df)
plan = plan_milp(free, GOALS)

tab = timetable(df, plan, GOALS, range(0, 7), free)
pd.set_option("display.max_colwidth", 80)
print("[1주차 일정표 - AI 제안]")
print(tab.to_string(index=False))

print("\n[알림 문구 예시]")
for d in (1, 3, 6):
    print(notify(df, plan, GOALS, rest, d, conf), "\n")

# 간트 그래프: 가로축은 하루 중 시각, 세로축은 날짜 (라벨은 영문)
COL = {"routine": "#D0D4DA", "duty": "#C0392B", "training": "#E08A1E", "leave": "#8E9BAE",
       "G1": "#1F3864", "G2": "#2E8B57", "G3": "#7B5EA7"}
EN = {"routine": "Routine", "duty": "Duty", "training": "Training", "leave": "Leave",
      "G1": "Study (certificate)", "G2": "Fitness", "G3": "English words"}


def pieces(s, e):
    out = []
    while s < e:
        d = s // 1440
        end = min(e, (d + 1) * 1440)
        out.append((d, s - d * 1440, end - s))
        s = end
    return out


fig, ax = plt.subplots(figsize=(10, 5.5))
for r in df.itertuples():
    for d, x, w in pieces(r.start, r.end):
        ax.broken_barh([(x / 60, w / 60)], (d - 0.4, 0.8), color=COL[r.kind], edgecolor="white", linewidth=0.4)
for g, s, e in plan:
    for d, x, w in pieces(s, e):
        ax.broken_barh([(x / 60, w / 60)], (d - 0.4, 0.8), color=COL[g], edgecolor="white", linewidth=0.4)
ax.set_xlim(0, 24); ax.set_ylim(13.6, -0.6)
ax.set_yticks(range(14)); ax.set_yticklabels([f"{['Mon','Tue','Wed','Thu','Fri','Sat','Sun'][d % 7]} {(START + dt.timedelta(days=d)):%m/%d}" for d in range(14)])
ax.set_xticks(range(0, 25, 3)); ax.set_xlabel("hour of day"); ax.set_title("Two-week schedule with suggested personal blocks (virtual unit)")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=COL[k]) for k in COL], labels=[EN[k] for k in COL],
          loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=4, fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

texts = [notify(df, plan, GOALS, rest, d, conf) for d in range(14)]
print(f"알림 14일분 생성: 휴식 부족 경고 {sum('휴식' in x for x in texts)}건, 충돌 확인 요청 {sum('확인 필요: ' in x for x in texts)}건,",
      "모든 문구에 확정 주체(분대장·행정반) 안내 포함:", all("분대장·행정반" in x for x in texts))


---

## 🎯 미션 09. 과부하 상황 시험하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **근무가 늘거나 목표가 많은 과부하 상황을 시험하고 조정안을 제안합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **근무 6회 추가, 목표 2배, 둘 다인 상황을 만들어 가용 시간, 요구 시간, 부하율, 달성률, 하루 부하(10시간 초과일)를 비교 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `과부하 시험 비교표, 조정안, 조정 후 결과` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 과부하 시험 — 과부하를 수치로 경고하고 조정안을 제안해 효과를 확인했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(과부하 시험 비교표, 조정안, 조정 후 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""근무가 늘거나 개인 목표가 너무 많은 과부하 상황을 만들어 시험하고, 부족분과 하루 부하를 경고하며 조정안을 제안한다."""
import re
import copy
import datetime as dt
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import coo_matrix

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 2주(11/02~11/15) 일정 자료 (실제 부대·인원·규정과 무관)
# 시각은 "일차(0=11/02 월요일) + HH:MM"으로 표기하며, 24시를 넘기면 26:00처럼 씁니다.
# ---------------------------------------------------------------------------
START = dt.date(2026, 11, 2)
DOW = "월화수목금토일"
DAY_END = "21:00"          # 개인 시간 계획은 저녁 점호(21:00) 전까지만 다룬다


def t(day, hhmm):
    # [TODO : 학생 실습 9] 과부하 상황 시험하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def E(eid, name, kind, day, s, e, end_day=None, sub=""):
    # [TODO : 학생 실습 9] 과부하 상황 시험하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_routine():
    ev = []
    for d in range(14):
        wd = d % 7
        if wd < 5:
            items = [("기상·점호", "fixed", "06:30", "07:00"), ("조식", "fixed", "07:00", "08:00"),
                     ("청소·정비", "fixed", "08:00", "09:00"), ("오전 일과", "work", "09:00", "12:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("오후 일과", "work", "13:00", "17:30"),
                     ("석식", "fixed", "17:30", "18:30"), ("저녁 점호", "fixed", "21:00", "21:30")]
        else:
            items = [("기상·점호", "fixed", "07:30", "08:00"), ("조식", "fixed", "08:00", "09:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("석식", "fixed", "17:30", "18:30"),
                     ("저녁 점호", "fixed", "21:00", "21:30")]
            if wd == 5:
                items.insert(2, ("주말 청소·정비", "work", "09:00", "10:30"))
        for k, (nm, sub, s, e) in enumerate(items):
            ev.append(E(f"R{d:02d}{k}", nm, "routine", d, s, e, sub=sub))
    return ev


EVENTS = make_routine() + [
    # 개인 근무표 (가상): 불침번·경계근무·당직 보조
    E("D01", "불침번", "duty", 0, "26:00", "28:00", sub="불침번"),
    E("D02", "불침번", "duty", 2, "22:00", "24:00", sub="불침번"),
    E("D03", "경계근무", "duty", 3, "16:00", "18:00", sub="경계근무"),
    E("D04", "불침번", "duty", 4, "26:00", "28:00", sub="불침번"),
    E("D05", "당직 보조", "duty", 6, "08:00", "32:00", sub="당직"),
    E("D06", "불침번", "duty", 8, "22:00", "24:00", sub="불침번"),
    E("D07", "불침번", "duty", 9, "22:00", "24:00", sub="불침번"),
    # 교육·훈련 일정
    E("T01", "정훈교육", "training", 1, "14:00", "16:00"),
    E("T02", "사이버 보안 교육", "training", 2, "20:30", "21:30"),
    E("T03", "정신교육", "training", 3, "15:00", "17:00"),
    E("T04", "사격 훈련", "training", 4, "09:00", "12:00"),
    E("T05", "동계 피복 보급", "training", 4, "13:00", "17:00"),
    E("T06", "보안 점검 교육", "training", 7, "15:00", "17:00"),
    E("T07", "체력검정", "training", 8, "10:00", "12:00"),
    E("T08", "신체검사(의무실)", "training", 8, "10:30", "11:30"),
    E("T09", "독감 예방접종", "training", 9, "09:00", "10:30"),
    E("T10", "분대 전술훈련(오전)", "training", 10, "09:00", "12:00"),
    E("T11", "분대 전술훈련(오후)", "training", 10, "13:00", "17:30"),
    # 휴가 (금요일 오전부터 일요일 저녁까지)
    E("V01", "정기 휴가", "leave", 11, "09:00", "18:00", end_day=13),
]

# 근무 종류별 근무 후 최소 휴식 시간(시간) - 교육용 가상 기준
REST_RULE = {"불침번": 6, "당직": 12, "경계근무": 2}

# 개인 목표: priority 1(낮음)~5(높음), allow=하루 중 허용 시간대(분), pref=선호 시간대(분)
GOALS = [
    {"id": "G1", "name": "자격증 공부", "target": 720, "lens": [60, 90, 120], "day_cap": 180,
     "deadline": (10, "24:00"), "priority": 3, "allow": (540, 1260), "pref": (1110, 1260)},
    {"id": "G2", "name": "체력단련", "target": 300, "lens": [60, 90], "day_cap": 90,
     "deadline": (13, "24:00"), "priority": 2, "allow": (360, 1260), "pref": None},
    {"id": "G3", "name": "영어 단어 학습", "target": 240, "lens": [30, 60], "day_cap": 60,
     "deadline": (13, "24:00"), "priority": 1, "allow": (540, 1260), "pref": None},
]

# 일정 변경 공지 (가상): 원래 일정을 옮기거나 늘리거나 취소한다
CHANGES = [
    {"id": "S01", "from": "교육훈련과", "title": "사격 훈련 연기(11/06 -> 11/13)", "op": "move", "target": "T04",
     "day": 11, "start": "09:00", "end": "12:00"},
    {"id": "S02", "from": "교육훈련과", "title": "체력검정 일정 변경(11/10 -> 11/11)", "op": "move", "target": "T07",
     "day": 9, "start": "10:00", "end": "12:00"},
    {"id": "S03", "from": "교육훈련과", "title": "분대 전술훈련 오후 일정 야간 연장", "op": "extend", "target": "T11", "end": "21:00"},
    {"id": "S04", "from": "작전과", "title": "경계근무 취소", "op": "cancel", "target": "D03"},
]

KINDS = {"routine": "부대 일과", "duty": "근무", "training": "교육·훈련", "leave": "휴가"}


def validate(ev):
    """일정 한 건의 형식 오류 목록을 돌려준다(빈 목록이면 정상)."""
    errs = [f"{k} 누락" for k in ("id", "name", "kind", "day", "start", "end") if ev.get(k) in (None, "")]
    if ev.get("kind") not in KINDS:
        errs.append("종류 오류")
    for k in ("start", "end"):
        if ev.get(k) and not re.match(r"^\d{1,2}:[0-5]\d$", str(ev[k])):
            errs.append(f"{k} 시간 형식 오류")
    if not errs:
        if not 0 <= ev["day"] < 14:
            errs.append("기간(0~13일) 밖")
        elif t(ev.get("end_day", ev["day"]), ev["end"]) <= t(ev["day"], ev["start"]):
            errs.append("종료가 시작보다 빠르거나 같음")
    return errs


def to_df(raw):
    """일정 목록을 '시작·종료 = 11/02 00:00부터의 분' 표로 바꾼다."""
    rows = []
    for ev in raw:
        s, e = t(ev["day"], ev["start"]), t(ev.get("end_day", ev["day"]), ev["end"])
        rows.append({"id": ev["id"], "name": ev["name"], "kind": ev["kind"], "sub": ev.get("sub", ""),
                     "start": s, "end": e, "dur": e - s})
    return pd.DataFrame(rows).sort_values(["start", "end"]).reset_index(drop=True)


def fmt(x):
    d, r = divmod(int(x), 1440)
    day = START + dt.timedelta(days=d)
    return f"{day.month:02d}/{day.day:02d}({DOW[d % 7]}) {r // 60:02d}:{r % 60:02d}"

def check_rest(df):
    """근무가 끝난 뒤 다음 일과·교육·근무가 시작될 때까지의 휴식 시간이 기준 이상인지 점검한다."""
    work = df[(df["kind"].isin(["duty", "training"])) | ((df["kind"] == "routine") & (df["sub"] == "work"))]
    rows = []
    for d in df[df["kind"] == "duty"].itertuples():
        need = REST_RULE.get(d.sub, 0) * 60
        nxt = work[work["start"] >= d.end]
        if nxt.empty:
            continue
        n = nxt.iloc[0]
        rest = int(n["start"] - d.end)
        rows.append({"근무": f"{d.id} {d.name}", "종료": fmt(d.end), "다음 일정": n["name"], "다음 시작": fmt(n["start"]),
                     "휴식(시간)": round(rest / 60, 1), "기준(시간)": need / 60,
                     "판정": "적정" if rest >= need else "위반", "부족(분)": max(0, need - rest), "id": d.id})
    return pd.DataFrame(rows)

def day_window(d):
    return (t(d, "06:30") if d % 7 < 5 else t(d, "07:30")), t(d, DAY_END)


def merge(iv):
    out = []
    for s, e in sorted(iv):
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out


def free_intervals(df, min_len=30, protect=True):
    """일과·근무·교육·휴가를 모두 뺀 빈 시간 구간. protect=True면 근무 후 휴식 구간도 뺀다."""
    busy = [(r.start, r.end) for r in df.itertuples()]
    if protect:
        busy += [(r.end, r.end + REST_RULE.get(r.sub, 0) * 60) for r in df[df["kind"] == "duty"].itertuples()]
    busy = merge(busy)
    free = []
    for d in range(14):
        lo, hi = day_window(d)
        cur = lo
        for s, e in busy:
            if e <= cur or s >= hi:
                continue
            if s > cur:
                free.append((cur, min(s, hi)))
            cur = max(cur, e)
        if cur < hi:
            free.append((cur, hi))
    return [(s, e) for s, e in free if e - s >= min_len]

SLOT = 30                                   # 계획은 30분 칸 단위
PW = {1: 1.0, 2: 2.0, 3: 3.0, 4: 4.0, 5: 5.0}   # 우선순위 가중치


def cand_blocks(free, g):
    """목표 g가 들어갈 수 있는 모든 후보 블록(시작, 종료)."""
    dl = t(*g["deadline"])
    out = []
    for s, e in free:
        d = s // 1440
        a, b = max(s, d * 1440 + g["allow"][0]), min(e, d * 1440 + g["allow"][1], dl)
        for st in range(-(-a // SLOT) * SLOT, b, SLOT):
            for L in g["lens"]:
                if st + L <= b:
                    out.append((g["id"], st, st + L))
    return out


def block_value(g, st, en, mode="priority"):
    L = en - st
    v = (PW[g["priority"]] if mode == "priority" else 1.0) * L
    if g["pref"] and g["pref"][0] <= st % 1440 and (en - 1) % 1440 < g["pref"][1]:
        v += 0.2 * L                        # 선호 시간대(저녁) 가산
    return v - 5 - 0.0005 * st              # 블록 수는 적게, 같은 값이면 이른 시각 우선


def plan_greedy(free, goals, day_total_cap=240):
    """탐욕: 우선순위 높은 목표부터, 이른 빈 시간부터 차례로 채운다."""
    ok = set()
    for s, e in free:
        ok.update(range(s // SLOT, e // SLOT))
    order = sorted(goals, key=lambda g: (-g["priority"], t(*g["deadline"])))
    used, blocks = {}, []
    for g in order:
        remain, dl = g["target"], t(*g["deadline"]) // SLOT
        lo, hi, mn, mx = g["allow"][0] // SLOT, g["allow"][1] // SLOT, min(g["lens"]), max(g["lens"])
        for d in range(14):
            cap = min(g["day_cap"], day_total_cap - used.get(d, 0), remain)
            k, end = d * 48 + lo, min(d * 48 + hi, dl)
            while remain > 0 and cap >= mn and k < end:
                if k not in ok:
                    k += 1
                    continue
                run = 0
                while k + run < end and (k + run) in ok:
                    run += 1
                L = min(mx, run * SLOT, cap) // SLOT * SLOT
                if L >= mn:
                    blocks.append((g["id"], k * SLOT, k * SLOT + L))
                    ok.difference_update(range(k, k + L // SLOT))
                    cap -= L; remain -= L; used[d] = used.get(d, 0) + L
                    k += L // SLOT
                else:
                    k += run
    return sorted(blocks, key=lambda b: b[1])


def plan_milp(free, goals, mode="priority", day_total_cap=240, prev=None, bonus=0.0):
    """최적화: 후보 블록 중 겹치지 않게 고르되 목표 시간·하루 상한·마감을 지키며 가치 합을 최대로 한다."""
    G = {g["id"]: g for g in goals}
    cands = [c for g in goals for c in cand_blocks(free, g)]
    if not cands:
        return []
    groups = {}
    for i, (gid, s, e) in enumerate(cands):
        for k in range(s // SLOT, e // SLOT):
            groups.setdefault(("slot", k), (1, []))[1].append((i, 1))
        groups.setdefault(("goal", gid), (G[gid]["target"], []))[1].append((i, e - s))
        groups.setdefault(("gday", gid, s // 1440), (G[gid]["day_cap"], []))[1].append((i, e - s))
        groups.setdefault(("day", s // 1440), (day_total_cap, []))[1].append((i, e - s))
    r, c, v, ub = [], [], [], []
    for row, (cap, items) in enumerate(groups.values()):
        ub.append(cap)
        for i, coef in items:
            r.append(row); c.append(i); v.append(coef)
    A = coo_matrix((v, (r, c)), shape=(len(ub), len(cands)))
    val = np.array([block_value(G[g], s, e, mode) + (bonus * (e - s) if prev and (g, s, e) in prev else 0)
                    for g, s, e in cands])
    res = milp(-val, constraints=LinearConstraint(A, -np.inf, np.array(ub, float)),
               integrality=np.ones(len(cands)), bounds=Bounds(0, 1), options={"time_limit": 20})
    if res.x is None:
        return []
    return sorted([cands[i] for i in range(len(cands)) if res.x[i] > 0.5], key=lambda b: b[1])


def evaluate(blocks, goals):
    rows = []
    for g in goals:
        bs = [b for b in blocks if b[0] == g["id"]]
        got = sum(e - s for _, s, e in bs)
        late = sum(1 for _, s, e in bs if e > t(*g["deadline"]))
        rows.append({"목표": g["name"], "요구(분)": g["target"], "배정(분)": got, "달성률": f"{got / g['target']:.0%}",
                     "블록 수": len(bs), "마감 초과": late,
                     "판정": "목표 달성" if got >= g["target"] and late == 0 else "미달"})
    return pd.DataFrame(rows)


def blocks_outside_free(blocks, free):
    """빈 시간 밖(다른 일정이나 휴식 보호 구간과 겹침)에 놓인 블록 수"""
    return sum(1 for _, s, e in blocks if not any(fs <= s and e <= fe for fs, fe in free))

def classify(a, b):
    """겹친 두 일정의 성격: hard(실제 충돌) / soft(식사·점호 침범) / replace(일과 대체) / exempt(근무·휴가자 일과 열외)"""
    if a["kind"] != "routine" and b["kind"] != "routine":
        return "hard"
    r, o = (a, b) if a["kind"] == "routine" else (b, a)
    if o["kind"] == "routine":
        return "hard"
    if o["kind"] in ("duty", "leave"):
        return "exempt"
    return "replace" if r["sub"] == "work" else "soft"


def find_conflicts(df):
    """시작 시각순으로 정렬된 일정에서 겹치는 쌍을 찾는다(끝난 일정 뒤로는 비교하지 않음)."""
    rows = df.to_dict("records")
    out = []
    for i, a in enumerate(rows):
        for b in rows[i + 1:]:
            if b["start"] >= a["end"]:
                break
            out.append({"일정A": f"{a['id']} {a['name']}", "일정B": f"{b['id']} {b['name']}",
                        "겹침(분)": min(a["end"], b["end"]) - b["start"], "시작": b["start"],
                        "구분": classify(a, b)})
    return pd.DataFrame(out, columns=["일정A", "일정B", "겹침(분)", "시작", "구분"])

def day_label(d):
    x = START + dt.timedelta(days=d)
    return f"{x.month:02d}/{x.day:02d}({DOW[d % 7]})"


def rel(x, lo):
    v = int(x - lo)
    return f"{v // 60:02d}:{v % 60:02d}"


def timetable(df, blocks, goals, days, free):
    """하루 단위 일정표: 근무·교육·휴가와 개인 목표 배정, 남는 빈 시간."""
    names = {g["id"]: g["name"] for g in goals}
    rows = []
    for d in days:
        lo, hi = d * 1440, (d + 1) * 1440
        part = df[(df["end"] > lo) & (df["start"] < hi) & (df["kind"] != "routine")]
        fixed = [f"{r.name} {rel(max(r.start, lo), lo)}~{rel(min(r.end, hi), lo)}" for r in part.itertuples()]
        mine = [b for b in blocks if lo <= b[1] < hi]
        pers = [f"{names[g]} {rel(s, lo)}~{rel(e, lo)}" for g, s, e in mine]
        free_min = sum(min(e, hi) - max(s, lo) for s, e in free if e > lo and s < hi) - sum(e - s for _, s, e in mine)
        rows.append({"날짜": day_label(d), "근무·교육·휴가": ", ".join(fixed) or "-",
                     "개인 목표 배정": ", ".join(pers) or "-", "남는 빈 시간(분)": free_min})
    return pd.DataFrame(rows)


def notify(df, blocks, goals, rest, d, conf=None):
    """다음 날(d일차) 알림 문구. 모든 문구는 AI의 제안이며 확정은 사람이 한다."""
    names = {g["id"]: g["name"] for g in goals}
    lo, hi = d * 1440, (d + 1) * 1440
    lines = [f"[일정 알림 - AI 제안] {day_label(d)} 일정"]
    for r in df[(df["start"] >= lo) & (df["start"] < hi) & (df["kind"] == "duty")].itertuples():
        w = rest[rest["id"] == r.id] if len(rest) else rest
        tail = ""
        if len(w) and w.iloc[0]["판정"] == "위반":
            tail = f" (근무 후 휴식 {w.iloc[0]['휴식(시간)']}시간으로 기준 {w.iloc[0]['기준(시간)']:.0f}시간보다 짧음 - 분대장 확인 필요)"
        lines.append(f"- 근무: {r.name} {rel(r.start, lo)}~{rel(r.end, lo) if r.end <= hi else '익일 ' + rel(r.end, hi)}{tail}")
    for r in df[(df["end"] > lo) & (df["start"] < hi) & (df["kind"].isin(["training", "leave"]))].itertuples():
        lines.append(f"- {KINDS[r.kind]}: {r.name} {rel(max(r.start, lo), lo)}~{rel(min(r.end, hi), lo)}")
    mine = [b for b in blocks if lo <= b[1] < hi]
    for g, s, e in mine:
        lines.append(f"- 개인 목표 제안: {rel(s, lo)}~{rel(e, lo)} {names[g]} ({e - s}분)")
    if conf is not None and len(conf):
        for c in conf[(conf["구분"] == "hard") & (conf["시작"] >= lo) & (conf["시작"] < hi)].to_dict("records"):
            lines.append(f"- 확인 필요: {c['일정A']} / {c['일정B']} {c['겹침(분)']}분 겹침")
    if len(lines) == 1:
        lines.append("- 특이 일정 없음")
    lines.append("※ 근무 편성과 일정 확정은 분대장·행정반이 합니다. 이 알림은 제안입니다.")
    return "\n".join(lines)


def apply_changes(raw, changes):
    """변경 공지를 일정 목록에 반영한다. 대상 일정이 없거나 형식이 잘못된 공지는 건너뛴다."""
    ev, log = {e["id"]: dict(e) for e in raw}, []
    for c in changes:
        e = ev.get(c.get("target"))
        if e is None:
            log.append(f"{c['id']}: 대상 일정 없음 - 반영 안 함")
        elif c["op"] == "cancel":
            del ev[c["target"]]; log.append(f"{c['id']}: {e['name']} 취소")
        elif c["op"] == "move":
            e.update(day=c["day"], end_day=c["day"], start=c["start"], end=c["end"]); log.append(f"{c['id']}: {e['name']} 이동 -> {day_label(c['day'])} {c['start']}~{c['end']}")
        elif c["op"] == "extend":
            e["end"] = c["end"]; log.append(f"{c['id']}: {e['name']} 종료 {c['end']}로 연장")
        else:
            log.append(f"{c['id']}: 알 수 없는 변경 유형 - 반영 안 함")
    return list(ev.values()), log

DAY_CAP = 240   # 하루 개인 목표 상한(분)


def scenario(events, goals):
    df = to_df(events)
    free = free_intervals(df)
    bl = plan_milp(free, goals)
    cap = sum(min(DAY_CAP, sum(e - s for s, e in free if s // 1440 == d)) for d in range(14))
    need = sum(g["target"] for g in goals)
    got = sum(e - s for _, s, e in bl)
    ratio = need / cap
    # 하루 부하: 근무+교육·훈련+개인 목표 배정 시간(시간)
    load = np.zeros(14)
    for r in df[df["kind"].isin(["duty", "training"])].itertuples():
        for d in range(14):
            load[d] += max(0, min(r.end, (d + 1) * 1440) - max(r.start, d * 1440)) / 60
    for _, s, e in bl:
        load[s // 1440] += (e - s) / 60
    return {"가용(분)": cap, "요구(분)": need, "배정(분)": got, "부하율": f"{ratio:.0%}",
            "판정": "과부하" if ratio > 1 else ("빠듯함" if ratio > 0.7 else "여유"),
            "달성률": f"{got / need:.0%}", "과로일(10시간 초과)": int((load > 10).sum()),
            "충돌": blocks_outside_free(bl, free)}, bl, df


extra_duty = [E(f"X{d}", "경계근무(추가)", "duty", d, "18:30", "21:00", sub="경계근무") for d in (0, 1, 3, 4, 7, 8)]
double = copy.deepcopy(GOALS)
for g in double:
    g["target"] *= 2

cases = {"기본": (EVENTS, GOALS), "근무 6회 추가": (EVENTS + extra_duty, GOALS),
         "목표 2배": (EVENTS, double), "근무 추가 + 목표 2배": (EVENTS + extra_duty, double)}
rows, plans = [], {}
for name, (ev, gl) in cases.items():
    r, bl, df = scenario(ev, gl)
    rows.append({"상황": name, **r})
    plans[name] = (bl, gl)
print(pd.DataFrame(rows).to_string(index=False))

name = "근무 추가 + 목표 2배"
bl, gl = plans[name]
print(f"\n[{name}] 목표별 결과")
print(evaluate(bl, gl)[["목표", "요구(분)", "배정(분)", "달성률", "판정"]].to_string(index=False))

print("\n[조정안 제안 - 확정은 사람이]")
ev_tab = evaluate(bl, gl)
for g, r in zip(gl, ev_tab.to_dict("records")):
    if r["판정"] == "미달":
        short = r["요구(분)"] - r["배정(분)"]
        print(f"- {g['name']}: {short}분 부족 -> 목표를 {r['배정(분)']}분으로 줄이거나 마감을 연장하는 방안 (본인·분대장 협의)")
print("- 추가 근무가 몰린 저녁:", ", ".join(day_label(e['day']) for e in extra_duty), "-> 근무 교대 조정은 분대장·행정반이 결정")

# 조정안을 적용하면 과부하가 풀리는지 시험: 목표를 배정 가능한 만큼으로 줄여 다시 계산
fit = copy.deepcopy(gl)
for g, r in zip(fit, ev_tab.to_dict("records")):
    g["target"] = r["배정(분)"]
r2, bl2, _ = scenario(EVENTS + extra_duty, fit)
print(f"\n조정 후: 요구 {r2['요구(분)']}분, 배정 {r2['배정(분)']}분, 부하율 {r2['부하율']}, 판정 {r2['판정']}, 충돌 {r2['충돌']}")
print("목표 달성" if r2["달성률"] == "100%" and r2["충돌"] == 0 else "미달")


---

## 🎯 미션 10. 군 일정 관리 도우미 완성하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **전체 기능을 하나로 묶은 군 일정 관리 도우미를 완성하고 자체 점검합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **구조 검사, 충돌 탐지, 휴식 점검, 빈 시간, 개인 일정 최적 배정, 변경 공지 재계산, 알림을 한 클래스로 통합 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `군 일정 관리 도우미 코드, 자체 점검 결과, 검토용 프롬프트` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 통합·완성 — 도우미를 통합하고 자체 점검을 모두 통과했으며 근무 편성·확정은 사람이 한다는 절차를 명시했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(군 일정 관리 도우미 코드, 자체 점검 결과, 검토용 프롬프트)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""구조 검사, 충돌 탐지, 휴식 점검, 빈 시간, 개인 일정 최적 배정, 변경 공지 재계산, 일정표·알림을 하나로 묶은 '군 일정 관리 도우미'를 완성하고 자체 점검한다."""
import re
import copy
import datetime as dt
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import coo_matrix

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 2주(11/02~11/15) 일정 자료 (실제 부대·인원·규정과 무관)
# 시각은 "일차(0=11/02 월요일) + HH:MM"으로 표기하며, 24시를 넘기면 26:00처럼 씁니다.
# ---------------------------------------------------------------------------
START = dt.date(2026, 11, 2)
DOW = "월화수목금토일"
DAY_END = "21:00"          # 개인 시간 계획은 저녁 점호(21:00) 전까지만 다룬다


def t(day, hhmm):
    # [TODO : 학생 실습 10] 군 일정 관리 도우미 완성하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def E(eid, name, kind, day, s, e, end_day=None, sub=""):
    # [TODO : 학생 실습 10] 군 일정 관리 도우미 완성하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_routine():
    ev = []
    for d in range(14):
        wd = d % 7
        if wd < 5:
            items = [("기상·점호", "fixed", "06:30", "07:00"), ("조식", "fixed", "07:00", "08:00"),
                     ("청소·정비", "fixed", "08:00", "09:00"), ("오전 일과", "work", "09:00", "12:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("오후 일과", "work", "13:00", "17:30"),
                     ("석식", "fixed", "17:30", "18:30"), ("저녁 점호", "fixed", "21:00", "21:30")]
        else:
            items = [("기상·점호", "fixed", "07:30", "08:00"), ("조식", "fixed", "08:00", "09:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("석식", "fixed", "17:30", "18:30"),
                     ("저녁 점호", "fixed", "21:00", "21:30")]
            if wd == 5:
                items.insert(2, ("주말 청소·정비", "work", "09:00", "10:30"))
        for k, (nm, sub, s, e) in enumerate(items):
            ev.append(E(f"R{d:02d}{k}", nm, "routine", d, s, e, sub=sub))
    return ev


EVENTS = make_routine() + [
    # 개인 근무표 (가상): 불침번·경계근무·당직 보조
    E("D01", "불침번", "duty", 0, "26:00", "28:00", sub="불침번"),
    E("D02", "불침번", "duty", 2, "22:00", "24:00", sub="불침번"),
    E("D03", "경계근무", "duty", 3, "16:00", "18:00", sub="경계근무"),
    E("D04", "불침번", "duty", 4, "26:00", "28:00", sub="불침번"),
    E("D05", "당직 보조", "duty", 6, "08:00", "32:00", sub="당직"),
    E("D06", "불침번", "duty", 8, "22:00", "24:00", sub="불침번"),
    E("D07", "불침번", "duty", 9, "22:00", "24:00", sub="불침번"),
    # 교육·훈련 일정
    E("T01", "정훈교육", "training", 1, "14:00", "16:00"),
    E("T02", "사이버 보안 교육", "training", 2, "20:30", "21:30"),
    E("T03", "정신교육", "training", 3, "15:00", "17:00"),
    E("T04", "사격 훈련", "training", 4, "09:00", "12:00"),
    E("T05", "동계 피복 보급", "training", 4, "13:00", "17:00"),
    E("T06", "보안 점검 교육", "training", 7, "15:00", "17:00"),
    E("T07", "체력검정", "training", 8, "10:00", "12:00"),
    E("T08", "신체검사(의무실)", "training", 8, "10:30", "11:30"),
    E("T09", "독감 예방접종", "training", 9, "09:00", "10:30"),
    E("T10", "분대 전술훈련(오전)", "training", 10, "09:00", "12:00"),
    E("T11", "분대 전술훈련(오후)", "training", 10, "13:00", "17:30"),
    # 휴가 (금요일 오전부터 일요일 저녁까지)
    E("V01", "정기 휴가", "leave", 11, "09:00", "18:00", end_day=13),
]

# 근무 종류별 근무 후 최소 휴식 시간(시간) - 교육용 가상 기준
REST_RULE = {"불침번": 6, "당직": 12, "경계근무": 2}

# 개인 목표: priority 1(낮음)~5(높음), allow=하루 중 허용 시간대(분), pref=선호 시간대(분)
GOALS = [
    {"id": "G1", "name": "자격증 공부", "target": 720, "lens": [60, 90, 120], "day_cap": 180,
     "deadline": (10, "24:00"), "priority": 3, "allow": (540, 1260), "pref": (1110, 1260)},
    {"id": "G2", "name": "체력단련", "target": 300, "lens": [60, 90], "day_cap": 90,
     "deadline": (13, "24:00"), "priority": 2, "allow": (360, 1260), "pref": None},
    {"id": "G3", "name": "영어 단어 학습", "target": 240, "lens": [30, 60], "day_cap": 60,
     "deadline": (13, "24:00"), "priority": 1, "allow": (540, 1260), "pref": None},
]

# 일정 변경 공지 (가상): 원래 일정을 옮기거나 늘리거나 취소한다
CHANGES = [
    {"id": "S01", "from": "교육훈련과", "title": "사격 훈련 연기(11/06 -> 11/13)", "op": "move", "target": "T04",
     "day": 11, "start": "09:00", "end": "12:00"},
    {"id": "S02", "from": "교육훈련과", "title": "체력검정 일정 변경(11/10 -> 11/11)", "op": "move", "target": "T07",
     "day": 9, "start": "10:00", "end": "12:00"},
    {"id": "S03", "from": "교육훈련과", "title": "분대 전술훈련 오후 일정 야간 연장", "op": "extend", "target": "T11", "end": "21:00"},
    {"id": "S04", "from": "작전과", "title": "경계근무 취소", "op": "cancel", "target": "D03"},
]

KINDS = {"routine": "부대 일과", "duty": "근무", "training": "교육·훈련", "leave": "휴가"}


def validate(ev):
    """일정 한 건의 형식 오류 목록을 돌려준다(빈 목록이면 정상)."""
    errs = [f"{k} 누락" for k in ("id", "name", "kind", "day", "start", "end") if ev.get(k) in (None, "")]
    if ev.get("kind") not in KINDS:
        errs.append("종류 오류")
    for k in ("start", "end"):
        if ev.get(k) and not re.match(r"^\d{1,2}:[0-5]\d$", str(ev[k])):
            errs.append(f"{k} 시간 형식 오류")
    if not errs:
        if not 0 <= ev["day"] < 14:
            errs.append("기간(0~13일) 밖")
        elif t(ev.get("end_day", ev["day"]), ev["end"]) <= t(ev["day"], ev["start"]):
            errs.append("종료가 시작보다 빠르거나 같음")
    return errs


def to_df(raw):
    """일정 목록을 '시작·종료 = 11/02 00:00부터의 분' 표로 바꾼다."""
    rows = []
    for ev in raw:
        s, e = t(ev["day"], ev["start"]), t(ev.get("end_day", ev["day"]), ev["end"])
        rows.append({"id": ev["id"], "name": ev["name"], "kind": ev["kind"], "sub": ev.get("sub", ""),
                     "start": s, "end": e, "dur": e - s})
    return pd.DataFrame(rows).sort_values(["start", "end"]).reset_index(drop=True)


def fmt(x):
    d, r = divmod(int(x), 1440)
    day = START + dt.timedelta(days=d)
    return f"{day.month:02d}/{day.day:02d}({DOW[d % 7]}) {r // 60:02d}:{r % 60:02d}"

def check_rest(df):
    """근무가 끝난 뒤 다음 일과·교육·근무가 시작될 때까지의 휴식 시간이 기준 이상인지 점검한다."""
    work = df[(df["kind"].isin(["duty", "training"])) | ((df["kind"] == "routine") & (df["sub"] == "work"))]
    rows = []
    for d in df[df["kind"] == "duty"].itertuples():
        need = REST_RULE.get(d.sub, 0) * 60
        nxt = work[work["start"] >= d.end]
        if nxt.empty:
            continue
        n = nxt.iloc[0]
        rest = int(n["start"] - d.end)
        rows.append({"근무": f"{d.id} {d.name}", "종료": fmt(d.end), "다음 일정": n["name"], "다음 시작": fmt(n["start"]),
                     "휴식(시간)": round(rest / 60, 1), "기준(시간)": need / 60,
                     "판정": "적정" if rest >= need else "위반", "부족(분)": max(0, need - rest), "id": d.id})
    return pd.DataFrame(rows)

def day_window(d):
    return (t(d, "06:30") if d % 7 < 5 else t(d, "07:30")), t(d, DAY_END)


def merge(iv):
    out = []
    for s, e in sorted(iv):
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out


def free_intervals(df, min_len=30, protect=True):
    """일과·근무·교육·휴가를 모두 뺀 빈 시간 구간. protect=True면 근무 후 휴식 구간도 뺀다."""
    busy = [(r.start, r.end) for r in df.itertuples()]
    if protect:
        busy += [(r.end, r.end + REST_RULE.get(r.sub, 0) * 60) for r in df[df["kind"] == "duty"].itertuples()]
    busy = merge(busy)
    free = []
    for d in range(14):
        lo, hi = day_window(d)
        cur = lo
        for s, e in busy:
            if e <= cur or s >= hi:
                continue
            if s > cur:
                free.append((cur, min(s, hi)))
            cur = max(cur, e)
        if cur < hi:
            free.append((cur, hi))
    return [(s, e) for s, e in free if e - s >= min_len]

SLOT = 30                                   # 계획은 30분 칸 단위
PW = {1: 1.0, 2: 2.0, 3: 3.0, 4: 4.0, 5: 5.0}   # 우선순위 가중치


def cand_blocks(free, g):
    """목표 g가 들어갈 수 있는 모든 후보 블록(시작, 종료)."""
    dl = t(*g["deadline"])
    out = []
    for s, e in free:
        d = s // 1440
        a, b = max(s, d * 1440 + g["allow"][0]), min(e, d * 1440 + g["allow"][1], dl)
        for st in range(-(-a // SLOT) * SLOT, b, SLOT):
            for L in g["lens"]:
                if st + L <= b:
                    out.append((g["id"], st, st + L))
    return out


def block_value(g, st, en, mode="priority"):
    L = en - st
    v = (PW[g["priority"]] if mode == "priority" else 1.0) * L
    if g["pref"] and g["pref"][0] <= st % 1440 and (en - 1) % 1440 < g["pref"][1]:
        v += 0.2 * L                        # 선호 시간대(저녁) 가산
    return v - 5 - 0.0005 * st              # 블록 수는 적게, 같은 값이면 이른 시각 우선


def plan_greedy(free, goals, day_total_cap=240):
    """탐욕: 우선순위 높은 목표부터, 이른 빈 시간부터 차례로 채운다."""
    ok = set()
    for s, e in free:
        ok.update(range(s // SLOT, e // SLOT))
    order = sorted(goals, key=lambda g: (-g["priority"], t(*g["deadline"])))
    used, blocks = {}, []
    for g in order:
        remain, dl = g["target"], t(*g["deadline"]) // SLOT
        lo, hi, mn, mx = g["allow"][0] // SLOT, g["allow"][1] // SLOT, min(g["lens"]), max(g["lens"])
        for d in range(14):
            cap = min(g["day_cap"], day_total_cap - used.get(d, 0), remain)
            k, end = d * 48 + lo, min(d * 48 + hi, dl)
            while remain > 0 and cap >= mn and k < end:
                if k not in ok:
                    k += 1
                    continue
                run = 0
                while k + run < end and (k + run) in ok:
                    run += 1
                L = min(mx, run * SLOT, cap) // SLOT * SLOT
                if L >= mn:
                    blocks.append((g["id"], k * SLOT, k * SLOT + L))
                    ok.difference_update(range(k, k + L // SLOT))
                    cap -= L; remain -= L; used[d] = used.get(d, 0) + L
                    k += L // SLOT
                else:
                    k += run
    return sorted(blocks, key=lambda b: b[1])


def plan_milp(free, goals, mode="priority", day_total_cap=240, prev=None, bonus=0.0):
    """최적화: 후보 블록 중 겹치지 않게 고르되 목표 시간·하루 상한·마감을 지키며 가치 합을 최대로 한다."""
    G = {g["id"]: g for g in goals}
    cands = [c for g in goals for c in cand_blocks(free, g)]
    if not cands:
        return []
    groups = {}
    for i, (gid, s, e) in enumerate(cands):
        for k in range(s // SLOT, e // SLOT):
            groups.setdefault(("slot", k), (1, []))[1].append((i, 1))
        groups.setdefault(("goal", gid), (G[gid]["target"], []))[1].append((i, e - s))
        groups.setdefault(("gday", gid, s // 1440), (G[gid]["day_cap"], []))[1].append((i, e - s))
        groups.setdefault(("day", s // 1440), (day_total_cap, []))[1].append((i, e - s))
    r, c, v, ub = [], [], [], []
    for row, (cap, items) in enumerate(groups.values()):
        ub.append(cap)
        for i, coef in items:
            r.append(row); c.append(i); v.append(coef)
    A = coo_matrix((v, (r, c)), shape=(len(ub), len(cands)))
    val = np.array([block_value(G[g], s, e, mode) + (bonus * (e - s) if prev and (g, s, e) in prev else 0)
                    for g, s, e in cands])
    res = milp(-val, constraints=LinearConstraint(A, -np.inf, np.array(ub, float)),
               integrality=np.ones(len(cands)), bounds=Bounds(0, 1), options={"time_limit": 20})
    if res.x is None:
        return []
    return sorted([cands[i] for i in range(len(cands)) if res.x[i] > 0.5], key=lambda b: b[1])


def evaluate(blocks, goals):
    rows = []
    for g in goals:
        bs = [b for b in blocks if b[0] == g["id"]]
        got = sum(e - s for _, s, e in bs)
        late = sum(1 for _, s, e in bs if e > t(*g["deadline"]))
        rows.append({"목표": g["name"], "요구(분)": g["target"], "배정(분)": got, "달성률": f"{got / g['target']:.0%}",
                     "블록 수": len(bs), "마감 초과": late,
                     "판정": "목표 달성" if got >= g["target"] and late == 0 else "미달"})
    return pd.DataFrame(rows)


def blocks_outside_free(blocks, free):
    """빈 시간 밖(다른 일정이나 휴식 보호 구간과 겹침)에 놓인 블록 수"""
    return sum(1 for _, s, e in blocks if not any(fs <= s and e <= fe for fs, fe in free))

def classify(a, b):
    """겹친 두 일정의 성격: hard(실제 충돌) / soft(식사·점호 침범) / replace(일과 대체) / exempt(근무·휴가자 일과 열외)"""
    if a["kind"] != "routine" and b["kind"] != "routine":
        return "hard"
    r, o = (a, b) if a["kind"] == "routine" else (b, a)
    if o["kind"] == "routine":
        return "hard"
    if o["kind"] in ("duty", "leave"):
        return "exempt"
    return "replace" if r["sub"] == "work" else "soft"


def find_conflicts(df):
    """시작 시각순으로 정렬된 일정에서 겹치는 쌍을 찾는다(끝난 일정 뒤로는 비교하지 않음)."""
    rows = df.to_dict("records")
    out = []
    for i, a in enumerate(rows):
        for b in rows[i + 1:]:
            if b["start"] >= a["end"]:
                break
            out.append({"일정A": f"{a['id']} {a['name']}", "일정B": f"{b['id']} {b['name']}",
                        "겹침(분)": min(a["end"], b["end"]) - b["start"], "시작": b["start"],
                        "구분": classify(a, b)})
    return pd.DataFrame(out, columns=["일정A", "일정B", "겹침(분)", "시작", "구분"])

def day_label(d):
    x = START + dt.timedelta(days=d)
    return f"{x.month:02d}/{x.day:02d}({DOW[d % 7]})"


def rel(x, lo):
    v = int(x - lo)
    return f"{v // 60:02d}:{v % 60:02d}"


def timetable(df, blocks, goals, days, free):
    """하루 단위 일정표: 근무·교육·휴가와 개인 목표 배정, 남는 빈 시간."""
    names = {g["id"]: g["name"] for g in goals}
    rows = []
    for d in days:
        lo, hi = d * 1440, (d + 1) * 1440
        part = df[(df["end"] > lo) & (df["start"] < hi) & (df["kind"] != "routine")]
        fixed = [f"{r.name} {rel(max(r.start, lo), lo)}~{rel(min(r.end, hi), lo)}" for r in part.itertuples()]
        mine = [b for b in blocks if lo <= b[1] < hi]
        pers = [f"{names[g]} {rel(s, lo)}~{rel(e, lo)}" for g, s, e in mine]
        free_min = sum(min(e, hi) - max(s, lo) for s, e in free if e > lo and s < hi) - sum(e - s for _, s, e in mine)
        rows.append({"날짜": day_label(d), "근무·교육·휴가": ", ".join(fixed) or "-",
                     "개인 목표 배정": ", ".join(pers) or "-", "남는 빈 시간(분)": free_min})
    return pd.DataFrame(rows)


def notify(df, blocks, goals, rest, d, conf=None):
    """다음 날(d일차) 알림 문구. 모든 문구는 AI의 제안이며 확정은 사람이 한다."""
    names = {g["id"]: g["name"] for g in goals}
    lo, hi = d * 1440, (d + 1) * 1440
    lines = [f"[일정 알림 - AI 제안] {day_label(d)} 일정"]
    for r in df[(df["start"] >= lo) & (df["start"] < hi) & (df["kind"] == "duty")].itertuples():
        w = rest[rest["id"] == r.id] if len(rest) else rest
        tail = ""
        if len(w) and w.iloc[0]["판정"] == "위반":
            tail = f" (근무 후 휴식 {w.iloc[0]['휴식(시간)']}시간으로 기준 {w.iloc[0]['기준(시간)']:.0f}시간보다 짧음 - 분대장 확인 필요)"
        lines.append(f"- 근무: {r.name} {rel(r.start, lo)}~{rel(r.end, lo) if r.end <= hi else '익일 ' + rel(r.end, hi)}{tail}")
    for r in df[(df["end"] > lo) & (df["start"] < hi) & (df["kind"].isin(["training", "leave"]))].itertuples():
        lines.append(f"- {KINDS[r.kind]}: {r.name} {rel(max(r.start, lo), lo)}~{rel(min(r.end, hi), lo)}")
    mine = [b for b in blocks if lo <= b[1] < hi]
    for g, s, e in mine:
        lines.append(f"- 개인 목표 제안: {rel(s, lo)}~{rel(e, lo)} {names[g]} ({e - s}분)")
    if conf is not None and len(conf):
        for c in conf[(conf["구분"] == "hard") & (conf["시작"] >= lo) & (conf["시작"] < hi)].to_dict("records"):
            lines.append(f"- 확인 필요: {c['일정A']} / {c['일정B']} {c['겹침(분)']}분 겹침")
    if len(lines) == 1:
        lines.append("- 특이 일정 없음")
    lines.append("※ 근무 편성과 일정 확정은 분대장·행정반이 합니다. 이 알림은 제안입니다.")
    return "\n".join(lines)


def apply_changes(raw, changes):
    """변경 공지를 일정 목록에 반영한다. 대상 일정이 없거나 형식이 잘못된 공지는 건너뛴다."""
    ev, log = {e["id"]: dict(e) for e in raw}, []
    for c in changes:
        e = ev.get(c.get("target"))
        if e is None:
            log.append(f"{c['id']}: 대상 일정 없음 - 반영 안 함")
        elif c["op"] == "cancel":
            del ev[c["target"]]; log.append(f"{c['id']}: {e['name']} 취소")
        elif c["op"] == "move":
            e.update(day=c["day"], end_day=c["day"], start=c["start"], end=c["end"]); log.append(f"{c['id']}: {e['name']} 이동 -> {day_label(c['day'])} {c['start']}~{c['end']}")
        elif c["op"] == "extend":
            e["end"] = c["end"]; log.append(f"{c['id']}: {e['name']} 종료 {c['end']}로 연장")
        else:
            log.append(f"{c['id']}: 알 수 없는 변경 유형 - 반영 안 함")
    return list(ev.values()), log


class ScheduleAssistant:
    def __init__(self, raw, goals, prev=None):
        self.raw, self.goals = [e for e in raw if not validate(e)], goals
        self.rejected = [e.get("id", "?") for e in raw if validate(e)]
        self.df = to_df(self.raw)
        self.conf = find_conflicts(self.df)
        self.rest = check_rest(self.df)
        self.free = free_intervals(self.df)
        self.plan = plan_milp(self.free, goals, prev=set(prev) if prev else None, bonus=1.0 if prev else 0.0)

    def hard(self):
        c = self.conf[self.conf["구분"] == "hard"]
        return set(zip(c["일정A"], c["일정B"]))

    def apply(self, changes):
        raw, log = apply_changes(self.raw, changes)
        return ScheduleAssistant(raw, self.goals, prev=self.plan), log

    def load_ratio(self):
        cap = sum(min(240, sum(e - s for s, e in self.free if s // 1440 == d)) for d in range(14))
        return sum(g["target"] for g in self.goals) / cap

    def card(self, d):
        return notify(self.df, self.plan, self.goals, self.rest, d, self.conf)


t0 = dt.datetime.now()
bad_events = [E("X1", "시간 거꾸로", "duty", 2, "10:00", "09:00"), E("X2", "종류 오류", "회의", 2, "10:00", "11:00")]
A = ScheduleAssistant(EVENTS + bad_events, GOALS)
B, log = A.apply(CHANGES)

print("[1] 현재 일정 점검")
print(f"- 일정 {len(A.df)}건(형식 오류 {len(A.rejected)}건 차단: {A.rejected}), 실제 충돌 {len(A.hard())}건, 휴식 위반 {(A.rest['판정'] == '위반').sum()}건")
print(evaluate(A.plan, GOALS)[["목표", "요구(분)", "배정(분)", "판정"]].to_string(index=False))
print("\n[2] 변경 공지 4건 반영")
for x in log:
    print("-", x)
print(f"- 충돌 해소 {len(A.hard() - B.hard())}건, 새 충돌 {len(B.hard() - A.hard())}건: {sorted(B.hard() - A.hard())}")
kept = len(set(A.plan) & set(B.plan))
print(f"- 개인 계획 유지 {kept}/{len(A.plan)}블록")
print("\n[3] 알림 문구(변경 후)")
print(B.card(9))
elapsed = (dt.datetime.now() - t0).total_seconds()

heavy = ScheduleAssistant(EVENTS + [E(f"X{d}", "경계근무(추가)", "duty", d, "18:30", "21:00", sub="경계근무") for d in (0, 1, 3, 4, 7, 8)], GOALS)
checks = {
    "형식 오류 일정 차단(X1, X2)": A.rejected == ["X1", "X2"],
    "실제 충돌 2건 탐지": len(A.hard()) == 2,
    "근무 후 휴식 위반 3건 탐지": (A.rest["판정"] == "위반").sum() == 3,
    "개인 목표 3개 모두 달성·마감 준수": all(evaluate(A.plan, GOALS)["판정"] == "목표 달성"),
    "배정 블록이 일정·휴식 구간과 겹치지 않음": blocks_outside_free(A.plan, A.free) == 0 and blocks_outside_free(B.plan, B.free) == 0,
    "변경 공지 반영(해소 2건, 새 충돌 2건 탐지)": len(A.hard() - B.hard()) == 2 and len(B.hard() - A.hard()) == 2,
    "변경 후에도 목표 달성, 기존 계획 10블록 이상 유지": all(evaluate(B.plan, GOALS)["판정"] == "목표 달성") and kept >= 10,
    "과부하 상황 경고(부하율 100% 초과)": heavy.load_ratio() > 1 and A.load_ratio() <= 1,
    "알림에 확정 주체(분대장·행정반) 안내": all("분대장·행정반" in A.card(d) for d in range(14)),
    "처리 시간 5초 이내": elapsed < 5,
}
print()
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print(f"자체 점검 {sum(bool(v) for v in checks.values())}/{len(checks)} 통과")

print("\n[생성형 AI로 검토할 때 쓰는 프롬프트 예시]")
print("너는 행정반 근무자다. 아래 가상 일정 목록과 AI가 제안한 개인 일정을 보고, 근무 후 휴식이 부족하거나 교육과 겹치는 부분이 있으면 지적하라. "
      "일정을 임의로 확정하거나 새로 만들지 말고 확인할 질문만 써라. (일정은 교육용 가상 자료만 사용)")
print("\n※ 이 도우미는 일정 충돌과 개인 일정을 제안할 뿐입니다. 근무 편성·일정 확정은 분대장과 행정반이 하며, 실제 일정은 원문 공지와 지휘계통으로 확인합니다.")
